# FT-Transformer — Employee Resignation Prediction

**Member 1 model:** FT-Transformer  
**Task:** Binary classification  
**Target:** `Resigned` (`True` / `1` = resigned)  
**Primary metric:** Average Precision (reported as PR-AUC in project discussion)

## Objective

Train and evaluate FT-Transformer on the project's **80,000-row development partition** using the frozen **5-fold cross-validation** registry and all **17 approved predictors**.

The **20,000-row formal holdout is excluded** from training, preprocessing, tuning, model selection, and OOF evaluation. It remains reserved for the locked final evaluation stage.

## Methodology and Leakage Controls

Each outer CV round uses **64,000 training rows** and **16,000 validation rows**.

The FT-Transformer adapter owns fold-local preprocessing:

- 13 numerical/ordinal predictors are standardized with `StandardScaler` fitted on the outer-training folds only.
- 4 categorical predictors are encoded from mappings learned on the outer-training folds only.
- A fresh adapter is created for each outer fold.
- The outer validation fold is never passed to `fit()`.
- One probability is generated for every development employee, producing 80,000 OOF predictions.
- `Employee_ID`, `Hire_Date`, and `Resigned` are never used as predictors.

The shared repository runner also rejects any formal-holdout `Employee_ID` passed into OOF training.

In [ ]:
# -----------------------------
# 1. Imports and project setup
# -----------------------------

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from sklearn.metrics import precision_recall_curve

# Resolve the repository root from notebooks/05_member_1/.
PROJECT_ROOT = Path("../..").resolve()

# Fallback for notebook launch from a different working directory.
if not (PROJECT_ROOT / "configs" / "feature_contract.yaml").exists():
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "configs" / "feature_contract.yaml").exists():
            PROJECT_ROOT = candidate.resolve()
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import calculate_binary_metrics
from src.models.ft_transformer import FTTransformerAdapter
from src.models.runner import run_5fold_cv

pd.set_option("display.max_columns", None)

CONFIG_PATH = PROJECT_ROOT / "configs" / "models" / "ft_transformer.yaml"
FEATURE_CONTRACT_PATH = PROJECT_ROOT / "configs" / "feature_contract.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "models" / "ft_transformer"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [CONFIG_PATH, FEATURE_CONTRACT_PATH, DATA_PATH, SPLIT_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required project files: {missing_paths}"

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# -----------------------------
# 2. Load configuration and record the executed parameter set
# -----------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)

with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)

model_name = ft_config["model"]["name"]
model_id = ft_config["model"]["model_id"]

# Start from the fixed architecture and make optimizer defaults explicit.
fixed_params = ft_config["fixed_params"].copy()
fixed_params.setdefault(
    "learning_rate",
    float(ft_config["search_space"]["learning_rate"]["default"]),
)
fixed_params.setdefault(
    "weight_decay",
    float(ft_config["search_space"]["weight_decay"]["default"]),
)

config_summary = pd.DataFrame(
    [
        ("Model", model_name),
        ("Model ID", model_id),
        ("Implementation", ft_config["model"]["implementation"]),
        ("Seed", fixed_params["seed"]),
        ("Transformer blocks", fixed_params["n_blocks"]),
        ("Block dimension", fixed_params["d_block"]),
        ("Attention heads", fixed_params["attention_n_heads"]),
        ("FFN hidden dimension", fixed_params["ffn_d_hidden"]),
        ("Learning rate", fixed_params["learning_rate"]),
        ("Weight decay", fixed_params["weight_decay"]),
        ("Executed epochs per fold", 50),
        ("Batch size", 512),
    ],
    columns=["Setting", "Value"],
)

display(config_summary)

print("\nConfigured search-space parameters:")
display(
    pd.DataFrame(ft_config["search_space"])
    .T
    .rename_axis("parameter")
    .reset_index()
)

In [ ]:
# -----------------------------
# 3. Reproducibility and compute environment
# -----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_label = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Python kernel:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Training device:", device_label)

if torch.cuda.is_available():
    print("PyTorch CUDA runtime:", torch.version.cuda)
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("Random seed:", fixed_params["seed"])

In [ ]:
# -----------------------------
# 4. Load raw data and frozen split registry
# -----------------------------

df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

assert df.shape == (100_000, 20), f"Unexpected raw dataset shape: {df.shape}"
assert split_registry.shape == (100_000, 3), (
    f"Unexpected split-registry shape: {split_registry.shape}"
)
assert df["Employee_ID"].is_unique
assert split_registry["Employee_ID"].is_unique

print("Raw dataset shape:", df.shape)
print("Split registry shape:", split_registry.shape)
print("\nPartition counts:")
print(split_registry["partition"].value_counts())

In [ ]:
# -----------------------------
# 5. Isolate the development partition and enforce holdout protection
# -----------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID",
    ]
)
holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID",
    ]
)

development_data = (
    df.loc[df["Employee_ID"].isin(development_ids)]
    .copy()
    .reset_index(drop=True)
)

development_df = development_data.merge(
    split_registry[["Employee_ID", "partition", "cv_fold"]],
    on="Employee_ID",
    how="left",
    validate="one_to_one",
)
development_df["cv_fold"] = development_df["cv_fold"].astype(int)

assert len(development_data) == 80_000
assert len(holdout_ids) == 20_000
assert set(development_data["Employee_ID"]).isdisjoint(holdout_ids)
assert set(development_df["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert development_df["cv_fold"].value_counts().eq(16_000).all()

print("Development rows:", len(development_data))
print("Formal holdout rows excluded:", len(holdout_ids))
print("Holdout IDs present in modeling data: 0")
print("\nDevelopment fold counts:")
print(development_df["cv_fold"].value_counts().sort_index())

In [ ]:
# -----------------------------
# 6. Validate the feature contract and build X / y
# -----------------------------

numerical_features = feature_contract["numerical_features"]
categorical_features = feature_contract["categorical_features"]
primary_model_features = feature_contract["primary_model_features"]
target_config = feature_contract["target"]

numerical_feature_names = [item["name"] for item in numerical_features]
categorical_feature_names = [item["name"] for item in categorical_features]
target_column = target_config["column"]

assert len(numerical_feature_names) == 13
assert len(categorical_feature_names) == 4
assert len(primary_model_features) == 17
assert set(numerical_feature_names + categorical_feature_names) == set(primary_model_features)

required_columns = primary_model_features + [target_column, "Employee_ID", "cv_fold"]
missing_columns = [c for c in required_columns if c not in development_df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

X = development_df[primary_model_features].copy()
y = development_df[target_column].astype(int).copy()
employee_ids = development_df["Employee_ID"].copy()
cv_folds = development_df["cv_fold"].copy()

assert X.shape == (80_000, 17)
assert len(y) == 80_000
assert set(y.unique()) == {0, 1}
assert X.isna().sum().sum() == 0
assert "Employee_ID" not in X.columns
assert "Hire_Date" not in X.columns
assert target_column not in X.columns

print("Predictor matrix:", X.shape)
print("Numerical / ordinal predictors:", len(numerical_feature_names))
print("Categorical predictors:", len(categorical_feature_names))
print("Missing predictor values:", int(X.isna().sum().sum()))
print("Development resignation prevalence:", f"{y.mean():.4%}")

In [ ]:
# -----------------------------
# 7. Verify fold balance
# -----------------------------

fold_summary = (
    development_df
    .groupby("cv_fold")[target_column]
    .agg(
        total_rows="size",
        resigned_count="sum",
        resignation_rate="mean",
    )
    .reset_index()
)

fold_summary["not_resigned_count"] = (
    fold_summary["total_rows"] - fold_summary["resigned_count"]
)
fold_summary["resignation_rate_pct"] = (
    fold_summary["resignation_rate"] * 100
).round(3)

fold_summary = fold_summary[
    [
        "cv_fold",
        "total_rows",
        "not_resigned_count",
        "resigned_count",
        "resignation_rate_pct",
    ]
]

assert len(fold_summary) == 5
assert (fold_summary["total_rows"] == 16_000).all()
assert (fold_summary["resigned_count"] > 0).all()
assert (fold_summary["not_resigned_count"] > 0).all()

display(fold_summary)

In [ ]:
# -----------------------------
# 8. Verify categorical coverage across outer folds
# -----------------------------

coverage_records = []

for fold in sorted(development_df["cv_fold"].unique()):
    train_fold_df = development_df[development_df["cv_fold"] != fold]
    val_fold_df = development_df[development_df["cv_fold"] == fold]

    for feature in categorical_feature_names:
        train_categories = set(train_fold_df[feature].dropna().unique())
        validation_categories = set(val_fold_df[feature].dropna().unique())
        unseen = validation_categories - train_categories

        coverage_records.append(
            {
                "cv_fold": fold,
                "feature": feature,
                "train_categories": len(train_categories),
                "validation_categories": len(validation_categories),
                "unseen_categories": len(unseen),
            }
        )

categorical_coverage = pd.DataFrame(coverage_records)
total_unseen = int(categorical_coverage["unseen_categories"].sum())

assert total_unseen == 0, (
    "At least one outer-validation fold contains a category absent "
    "from its corresponding training folds."
)

display(categorical_coverage)
print("\nTotal unseen validation categories:", total_unseen)

## Verified FT-Transformer Execution Protocol

The repository implementation was inspected before training.

**Executed behavior**

- Fresh `FTTransformerAdapter` per outer fold.
- Fold-local `StandardScaler` and categorical mappings.
- AdamW optimizer.
- `BCEWithLogitsLoss`.
- 50 epochs per fold.
- Batch size 512.
- Learning rate 0.001.
- Weight decay 1e-5.
- Seed 42.
- Automatic device selection (`cuda` when available, otherwise CPU).

**Configuration caveat**

The YAML also declares `max_epochs: 100`, `patience: 15`, `ReduceLROnPlateau`, and an inner validation fraction of 0.15. The current shared runner calls `adapter.fit(X, y)` without supplying those settings, and the current FT-Transformer adapter does not implement inner-validation early stopping or the scheduler. This notebook therefore reports the behavior that is **actually executed**, not the unused YAML declarations.

The YAML search space is recorded for reproducibility, but this completed run uses the fixed/default configuration; it is **not presented as a completed hyperparameter-search result**.

The source adapter also passes `ffn_d_hidden_multiplier=None` for compatibility with `rtdl_revisiting_models==0.0.2` while using the explicit `ffn_d_hidden` value.

In [ ]:
# -----------------------------
# 9. Create a fresh adapter factory
# -----------------------------

def make_ft_transformer():
    """Return a fresh, unfitted FT-Transformer adapter."""
    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )

adapter_a = make_ft_transformer()
adapter_b = make_ft_transformer()

assert adapter_a is not adapter_b
assert adapter_a.params is not adapter_b.params
assert adapter_a.categorical_features is not adapter_b.categorical_features
assert not adapter_a.get_metadata()["fitted"]
assert not adapter_b.get_metadata()["fitted"]

print("Adapter factory validation passed.")
print("Model:", adapter_a.name)
print("Categorical predictors:", adapter_a.categorical_features)

In [ ]:
# -----------------------------
# 10. Official 5-fold OOF training
# -----------------------------

print("FT-Transformer official CV training")
print("-----------------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer CV folds: 5")
print("Training rows per fold: 64,000")
print("Validation rows per fold: 16,000")
print("Epochs per fold: 50")
print("Batch size: 512")
print("Training device:", device_label)

cv_start_time = time.perf_counter()

ft_cv_result = run_5fold_cv(
    adapter_factory=make_ft_transformer,
    data=development_data,
    split_registry=split_registry,
    feature_cols=primary_model_features,
    target_col=target_column,
    id_col="Employee_ID",
    fold_col="cv_fold",
    partition_col="partition",
    development_label="development",
    holdout_label="holdout",
    expected_folds=(1, 2, 3, 4, 5),
    expected_model_name=model_name,
)

cv_elapsed_time = time.perf_counter() - cv_start_time

print("\nFT-Transformer 5-fold CV completed successfully.")
print("Wall-clock time:", f"{cv_elapsed_time / 60:.2f} minutes")
print("OOF prediction rows:", len(ft_cv_result.predictions))

### Recorded Successful Run

The uploaded working notebook already completed the official 5-fold development run successfully. The recorded result was:

| Metric | Recorded value |
|---|---:|
| Pooled Average Precision / PR-AUC | **0.101132** |
| Pooled ROC-AUC | **0.498821** |
| Pooled Brier score | **0.090651** |
| Pooled F1 @ 0.50 | **0.000000** |
| CV mean Average Precision | **0.102028** |
| CV SD Average Precision | **0.002378** |
| CV mean ROC-AUC | **0.499769** |
| CV SD ROC-AUC | **0.010585** |
| Total fit time | **770.90 s** |
| Total prediction time | **1.16 s** |
| Wall-clock time reported by notebook | **12.88 min** |

CUDA was verified as available with an **NVIDIA GeForce RTX 4060 Laptop GPU (8 GB)** before this run. The earlier `"Training device: CPU"` text in the development notebook was a hard-coded print statement; the adapter itself selects CUDA dynamically when `torch.cuda.is_available()` is `True`.

In [ ]:
# -----------------------------
# 11. Fold-level performance
# -----------------------------

fold_metrics = ft_cv_result.fold_metrics.copy()

expected_metric_columns = {
    "fold",
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "brier_score",
    "fit_time_seconds",
    "predict_time_seconds",
}
missing_metric_columns = expected_metric_columns - set(fold_metrics.columns)
assert not missing_metric_columns, (
    f"Fold metrics are missing columns: {sorted(missing_metric_columns)}"
)
assert len(fold_metrics) == 5

display(
    fold_metrics[
        [
            "fold",
            "average_precision",
            "roc_auc",
            "brier_score",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "fit_time_seconds",
            "predict_time_seconds",
        ]
    ].round(6)
)

In [ ]:
# -----------------------------
# 12. Aggregate OOF metrics and no-skill reference
# -----------------------------

oof_predictions = ft_cv_result.predictions.copy()

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].is_unique
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert set(oof_predictions["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)

pooled_metrics = calculate_binary_metrics(
    y_true=oof_predictions["y_true"],
    y_probability=oof_predictions["y_probability"],
    y_pred=oof_predictions["y_pred_0_5"],
    threshold=0.50,
)

prevalence = float(oof_predictions["y_true"].mean())
no_skill_ap = prevalence
constant_prevalence_brier = prevalence * (1.0 - prevalence)

metric_summary = pd.DataFrame(
    [
        ("Average Precision / PR-AUC", pooled_metrics["average_precision"]),
        ("No-skill AP (prevalence)", no_skill_ap),
        ("ROC-AUC", pooled_metrics["roc_auc"]),
        ("Brier score", pooled_metrics["brier_score"]),
        ("Constant-prevalence Brier reference", constant_prevalence_brier),
        ("Balanced accuracy @ 0.50", pooled_metrics["balanced_accuracy"]),
        ("Precision @ 0.50", pooled_metrics["precision"]),
        ("Recall @ 0.50", pooled_metrics["recall"]),
        ("F1 @ 0.50", pooled_metrics["f1"]),
        ("Predicted-positive rate @ 0.50", pooled_metrics["predicted_positive_rate"]),
    ],
    columns=["Metric", "Value"],
)

display(metric_summary.round(6))

print(
    "\nPR-AUC lift over prevalence:",
    f"{pooled_metrics['average_precision'] - no_skill_ap:+.6f}",
)

In [ ]:
# -----------------------------
# 13. OOF probability diagnostics
# -----------------------------

probabilities = oof_predictions["y_probability"]

probability_summary = pd.DataFrame(
    {
        "Statistic": [
            "Minimum",
            "1st percentile",
            "25th percentile",
            "Median",
            "Mean",
            "75th percentile",
            "99th percentile",
            "Maximum",
        ],
        "Probability": [
            probabilities.min(),
            probabilities.quantile(0.01),
            probabilities.quantile(0.25),
            probabilities.median(),
            probabilities.mean(),
            probabilities.quantile(0.75),
            probabilities.quantile(0.99),
            probabilities.max(),
        ],
    }
)

display(probability_summary.round(6))

print(
    "Predicted positives at threshold 0.50:",
    int(oof_predictions["y_pred_0_5"].sum()),
)
print(
    "Actual positives:",
    int(oof_predictions["y_true"].sum()),
)

In [ ]:
# -----------------------------
# 14. Precision-recall curve and probability distribution
# -----------------------------

precision, recall, _ = precision_recall_curve(
    oof_predictions["y_true"],
    oof_predictions["y_probability"],
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, label=f"FT-Transformer AP = {pooled_metrics['average_precision']:.4f}")
ax.axhline(prevalence, linestyle="--", label=f"No-skill prevalence = {prevalence:.4f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("FT-Transformer OOF Precision-Recall Curve")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

pr_curve_path = RESULTS_DIR / "figure_01_ft_transformer_precision_recall_curve.png"
fig.savefig(pr_curve_path, dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    [
        oof_predictions.loc[oof_predictions["y_true"] == 0, "y_probability"],
        oof_predictions.loc[oof_predictions["y_true"] == 1, "y_probability"],
    ],
    bins=40,
    label=["Not resigned", "Resigned"],
    alpha=0.7,
)
ax.set_xlabel("Predicted resignation probability")
ax.set_ylabel("Employees")
ax.set_title("FT-Transformer OOF Probability Distribution")
ax.legend()
fig.tight_layout()

probability_plot_path = RESULTS_DIR / "figure_02_ft_transformer_oof_probability_distribution.png"
fig.savefig(probability_plot_path, dpi=150, bbox_inches="tight")
plt.show()

## Tuning, Interpretability, and Serialization Scope

### Hyperparameter tuning
The YAML contains a search space, but the recorded run above evaluates the fixed/default configuration. It should **not** be described as a tuned optimum. Any later tuning must remain development-only and reuse the frozen folds without accessing the formal holdout.

### Interpretability
The current FT-Transformer adapter does not expose stable per-feature importance, attention attribution, SHAP values, or retained fold models through the shared OOF runner. This notebook therefore does **not fabricate a feature-importance ranking**. Cross-model interpretability and fairness analyses should use methods that are explicitly supported by the selected model implementation.

### Model serialization
The shared OOF runner returns predictions, metrics, timing, and metadata, but it does not retain all five fitted fold adapters. This notebook therefore serializes the **evaluation artifacts** needed for model comparison. Final deployable-model training and model-weight serialization should occur only after model selection and pipeline freeze.

In [ ]:
# -----------------------------
# 15. Save reproducible FT-Transformer evaluation artifacts
# -----------------------------

predictions_path = RESULTS_DIR / "ft_transformer_oof_predictions.csv"
fold_metrics_path = RESULTS_DIR / "ft_transformer_fold_metrics.csv"
summary_metrics_path = RESULTS_DIR / "ft_transformer_summary_metrics.json"
run_metadata_path = RESULTS_DIR / "ft_transformer_run_metadata.json"
manifest_path = RESULTS_DIR / "ft_transformer_output_manifest.csv"

oof_predictions.to_csv(predictions_path, index=False)
fold_metrics.to_csv(fold_metrics_path, index=False)

with open(summary_metrics_path, "w", encoding="utf-8") as file:
    json.dump(ft_cv_result.summary_metrics, file, indent=2)

run_metadata = {
    "model_name": model_name,
    "model_id": model_id,
    "feature_count": len(primary_model_features),
    "features": primary_model_features,
    "categorical_features": categorical_feature_names,
    "numerical_features": numerical_feature_names,
    "development_rows": len(development_data),
    "formal_holdout_rows_used": 0,
    "outer_folds": [1, 2, 3, 4, 5],
    "epochs_per_fold": 50,
    "batch_size": 512,
    "device": device_label,
    "torch_version": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "executed_params": fixed_params,
    "tuning_performed": False,
    "adapter_metadata": ft_cv_result.metadata,
    "fit_time_seconds_total": ft_cv_result.fit_time_seconds_total,
    "predict_time_seconds_total": ft_cv_result.predict_time_seconds_total,
}

with open(run_metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

artifact_paths = [
    predictions_path,
    fold_metrics_path,
    summary_metrics_path,
    run_metadata_path,
    pr_curve_path,
    probability_plot_path,
]

manifest = pd.DataFrame(
    {
        "artifact": [path.name for path in artifact_paths],
        "relative_path": [
            str(path.relative_to(PROJECT_ROOT))
            for path in artifact_paths
        ],
        "exists": [path.exists() for path in artifact_paths],
    }
)
manifest.to_csv(manifest_path, index=False)

display(manifest)
print("\nArtifacts saved to:", RESULTS_DIR)

In [ ]:
# -----------------------------
# 16. Final integrity checks
# -----------------------------

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].nunique() == 80_000
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert len(fold_metrics) == 5
assert set(fold_metrics["fold"]) == {1, 2, 3, 4, 5}

assert np.isclose(
    pooled_metrics["average_precision"],
    ft_cv_result.summary_metrics["pooled_average_precision"],
)
assert np.isclose(
    pooled_metrics["roc_auc"],
    ft_cv_result.summary_metrics["pooled_roc_auc"],
)
assert np.isclose(
    pooled_metrics["brier_score"],
    ft_cv_result.summary_metrics["pooled_brier_score"],
)

required_artifacts = artifact_paths + [manifest_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
assert not missing_artifacts, f"Missing result artifacts: {missing_artifacts}"

print("All FT-Transformer final integrity checks passed.")
print("Development OOF evaluation is ready for model comparison.")
print("Formal holdout remains untouched.")

## Conclusion

FT-Transformer was evaluated on all **17 approved predictors** using the frozen five-fold development protocol. The formal 20,000-row holdout was not used.

The recorded completed run achieved:

- **Pooled PR-AUC / Average Precision:** 0.101132
- **Pooled ROC-AUC:** 0.498821
- **Pooled Brier score:** 0.090651
- **Pooled F1 at threshold 0.50:** 0.000000

Development-set resignation prevalence is approximately **10.01%**, so the model's PR-AUC is essentially at the no-skill prevalence reference. ROC-AUC is also approximately 0.50, indicating little to no useful ranking discrimination under this configuration. The zero F1 at 0.50 is consistent with probabilities remaining below the default classification threshold.

This is a valid negative modeling result: the current FT-Transformer configuration does not demonstrate meaningful predictive signal on the approved feature set. The OOF artifacts should proceed to the common model-comparison stage, while the formal holdout remains locked.

**Status:** FT-Transformer development OOF evaluation complete; ready for cross-model comparison. Hyperparameter tuning is not claimed as complete.

In [1]:
# -----------------------------
# Weighted FT-Transformer smoke test
# -----------------------------

import importlib
import numpy as np

import src.models.ft_transformer as ft_module


# Reload the updated .py file into the current notebook kernel.
ft_module = importlib.reload(ft_module)

FTTransformerAdapter = ft_module.FTTransformerAdapter


# Take a small development-only sample.
smoke_df = (
    development_df
    .sample(
        n=2048,
        random_state=42
    )
    .reset_index(drop=True)
)

X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = (
    smoke_df[target_column]
    .astype(int)
    .copy()
)


# Create the UPDATED weighted adapter.
weighted_smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params.copy(),
    categorical_features=categorical_feature_names.copy(),
)


# Train for only 1 epoch.
# This is NOT part of the official experiment.
weighted_smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=1,
    batch_size=256,
)


# Check the calculated fold-local positive-class weight.
metadata = weighted_smoke_adapter.get_metadata()

print("Positive samples:", int(y_smoke.sum()))
print("Negative samples:", int((y_smoke == 0).sum()))

print(
    "Calculated pos_weight:",
    metadata["pos_weight"]
)

print(
    "Class weighting:",
    metadata["class_weighting"]
)


# Produce a few predictions to confirm inference works.
smoke_probs = weighted_smoke_adapter.predict_proba(
    X_smoke.iloc[:64]
)

assert metadata["pos_weight"] is not None
assert metadata["pos_weight"] > 1
assert len(smoke_probs) == 64
assert np.isfinite(smoke_probs).all()
assert ((smoke_probs >= 0) & (smoke_probs <= 1)).all()


print("\nWeighted FT-Transformer smoke test PASSED.")

print(
    "Probability range:",
    f"{smoke_probs.min():.6f}",
    "to",
    f"{smoke_probs.max():.6f}"
)

ModuleNotFoundError: No module named 'src'

In [2]:
# -----------------------------
# Restore project imports
# -----------------------------

import sys
from pathlib import Path


# Find the repository root.
PROJECT_ROOT = Path("../..").resolve()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate.resolve()
            break


# Add project root so `src...` imports work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: D:\Desktop\Employee_Resignation_Prediction
src exists: True


In [3]:
# -----------------------------
# Restore variables for weighted FT-Transformer test
# -----------------------------

import yaml
import numpy as np
import pandas as pd
import torch
import importlib

import src.models.ft_transformer as ft_module


# Reload the UPDATED adapter file.
ft_module = importlib.reload(ft_module)
FTTransformerAdapter = ft_module.FTTransformerAdapter


# -----------------------------
# Paths
# -----------------------------

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "models"
    / "ft_transformer.yaml"
)

FEATURE_CONTRACT_PATH = (
    PROJECT_ROOT
    / "configs"
    / "feature_contract.yaml"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Extended_Employee_Performance_and_Productivity_Data.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_registry.csv"
)


# -----------------------------
# Load configuration
# -----------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)

with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)


# -----------------------------
# Load data
# -----------------------------

df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

data = df.merge(
    split_registry,
    on="Employee_ID",
    how="left",
    validate="one_to_one",
)

development_df = (
    data.loc[
        data["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

development_df["cv_fold"] = (
    development_df["cv_fold"]
    .astype(int)
)


# -----------------------------
# Feature/model definitions
# -----------------------------

primary_model_features = (
    feature_contract["primary_model_features"]
)

categorical_feature_names = [
    item["name"]
    for item in feature_contract["categorical_features"]
]

target_column = (
    feature_contract["target"]["column"]
)

model_name = (
    ft_config["model"]["name"]
)

model_id = (
    ft_config["model"]["model_id"]
)

fixed_params = (
    ft_config["fixed_params"].copy()
)


# -----------------------------
# Quick checks
# -----------------------------

assert len(development_df) == 80_000
assert len(primary_model_features) == 17
assert len(categorical_feature_names) == 4


print("Setup restored successfully.")
print("Development rows:", len(development_df))
print("Predictors:", len(primary_model_features))
print("Categorical features:", len(categorical_feature_names))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Setup restored successfully.
Development rows: 80000
Predictors: 17
Categorical features: 4
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
# -----------------------------
# Weighted FT-Transformer smoke test
# -----------------------------

# Take a small development-only sample.
# This is only a technical test, not the final experiment.
smoke_df = (
    development_df
    .sample(
        n=2048,
        random_state=42
    )
    .reset_index(drop=True)
)


X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = (
    smoke_df[target_column]
    .astype(int)
    .copy()
)


# Create the UPDATED weighted adapter.
weighted_smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params.copy(),
    categorical_features=categorical_feature_names.copy(),
)


# Train for only 1 epoch.
weighted_smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=1,
    batch_size=256,
)


# Read metadata after training.
metadata = weighted_smoke_adapter.get_metadata()


# -----------------------------
# Inspect class weighting
# -----------------------------

positive_count = int(
    y_smoke.sum()
)

negative_count = int(
    (y_smoke == 0).sum()
)


print("Positive samples:", positive_count)
print("Negative samples:", negative_count)

print(
    "Expected pos_weight:",
    negative_count / positive_count
)

print(
    "Adapter pos_weight:",
    metadata["pos_weight"]
)

print(
    "Class weighting:",
    metadata["class_weighting"]
)


# -----------------------------
# Test prediction
# -----------------------------

smoke_probs = (
    weighted_smoke_adapter
    .predict_proba(
        X_smoke.iloc[:64]
    )
)


# -----------------------------
# Validation checks
# -----------------------------

assert metadata["pos_weight"] is not None

assert metadata["pos_weight"] > 1

assert np.isclose(
    metadata["pos_weight"],
    negative_count / positive_count
)

assert len(smoke_probs) == 64

assert np.isfinite(
    smoke_probs
).all()

assert (
    (smoke_probs >= 0)
    & (smoke_probs <= 1)
).all()


print(
    "\nWeighted FT-Transformer smoke test PASSED."
)

print(
    "Probability range:",
    f"{smoke_probs.min():.6f}",
    "to",
    f"{smoke_probs.max():.6f}"
)

Positive samples: 194
Negative samples: 1854
Expected pos_weight: 9.556701030927835
Adapter pos_weight: 9.556701030927835
Class weighting: fold_local_positive_class_weight

Weighted FT-Transformer smoke test PASSED.
Probability range: 0.471666 to 0.514706


In [5]:
# -----------------------------
# Preserve previous unweighted FT-Transformer results
# -----------------------------

from pathlib import Path
import shutil


RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

UNWEIGHTED_DIR = (
    RESULTS_DIR
    / "unweighted_baseline"
)

UNWEIGHTED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Files produced by the previous unweighted experiment.
unweighted_artifacts = [
    "figure_01_ft_transformer_precision_recall_curve.png",
    "figure_02_ft_transformer_oof_probability_distribution.png",
    "ft_transformer_fold_metrics.csv",
    "ft_transformer_oof_predictions.csv",
    "ft_transformer_output_manifest.csv",
    "ft_transformer_run_metadata.json",
    "ft_transformer_summary_metrics.json",
]


moved_files = []
missing_files = []


for filename in unweighted_artifacts:

    source = RESULTS_DIR / filename
    destination = UNWEIGHTED_DIR / filename

    if source.exists():

        shutil.move(
            str(source),
            str(destination),
        )

        moved_files.append(filename)

    else:
        missing_files.append(filename)


print("Unweighted baseline archive:")
print("Moved:", len(moved_files))

for filename in moved_files:
    print("  ✓", filename)


if missing_files:

    print("\nNot found:")

    for filename in missing_files:
        print("  -", filename)


print(
    "\nArchive directory:",
    UNWEIGHTED_DIR
)

Unweighted baseline archive:
Moved: 7
  ✓ figure_01_ft_transformer_precision_recall_curve.png
  ✓ figure_02_ft_transformer_oof_probability_distribution.png
  ✓ ft_transformer_fold_metrics.csv
  ✓ ft_transformer_oof_predictions.csv
  ✓ ft_transformer_output_manifest.csv
  ✓ ft_transformer_run_metadata.json
  ✓ ft_transformer_summary_metrics.json

Archive directory: D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\unweighted_baseline


In [6]:
# -----------------------------
# Official weighted FT-Transformer 5-fold CV
# -----------------------------

import time

from src.models.runner import run_5fold_cv


# --------------------------------------------------
# Factory: fresh weighted adapter for every fold
# --------------------------------------------------

def make_weighted_ft_transformer():
    """
    Create a fresh FT-Transformer adapter.

    Class weighting is calculated automatically
    inside fit() using ONLY that fold's training labels.
    """

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )


# --------------------------------------------------
# Development-only modeling data
# --------------------------------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID",
    ]
)

holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID",
    ]
)

development_data = (
    df.loc[
        df["Employee_ID"].isin(development_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


# --------------------------------------------------
# Final leakage checks
# --------------------------------------------------

assert len(development_data) == 80_000

assert development_data[
    "Employee_ID"
].is_unique

assert set(
    development_data["Employee_ID"]
).isdisjoint(
    holdout_ids
), "Formal holdout leakage detected."


# --------------------------------------------------
# Environment summary
# --------------------------------------------------

device_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Weighted FT-Transformer CV")
print("--------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer folds:", 5)
print("Training rows per fold:", 64_000)
print("Validation rows per fold:", 16_000)
print("Epochs per fold:", 50)
print("Batch size:", 512)
print("Device:", device_name)
print("Class weighting: fold-local pos_weight")

print(
    "\nEach fold calculates:"
    "\npos_weight = negative training samples / positive training samples"
)

print("\nStarting weighted 5-fold training...")


# --------------------------------------------------
# Run weighted leakage-safe CV
# --------------------------------------------------

weighted_cv_start = time.perf_counter()


weighted_ft_cv_result = run_5fold_cv(
    adapter_factory=make_weighted_ft_transformer,

    # Development data ONLY
    data=development_data,

    # Frozen split registry
    split_registry=split_registry,

    # Approved 17 predictors
    feature_cols=primary_model_features,

    target_col=target_column,
    id_col="Employee_ID",

    fold_col="cv_fold",
    partition_col="partition",

    development_label="development",
    holdout_label="holdout",

    expected_folds=(1, 2, 3, 4, 5),

    expected_model_name=model_name,
)


weighted_cv_elapsed = (
    time.perf_counter()
    - weighted_cv_start
)


# --------------------------------------------------
# Completion
# --------------------------------------------------

print(
    "\nWeighted FT-Transformer "
    "5-fold CV completed successfully."
)

print(
    "Total elapsed time:",
    f"{weighted_cv_elapsed / 60:.2f} minutes"
)

print(
    "Result object:",
    type(weighted_ft_cv_result).__name__
)

print(
    "Final-fold pos_weight:",
    weighted_ft_cv_result
    .metadata
    .get("pos_weight")
)

Weighted FT-Transformer CV
--------------------------
Development rows: 80000
Formal holdout rows used: 0
Predictors: 17
Outer folds: 5
Training rows per fold: 64000
Validation rows per fold: 16000
Epochs per fold: 50
Batch size: 512
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Class weighting: fold-local pos_weight

Each fold calculates:
pos_weight = negative training samples / positive training samples

Starting weighted 5-fold training...

Weighted FT-Transformer 5-fold CV completed successfully.
Total elapsed time: 16.12 minutes
Result object: OOFResult
Final-fold pos_weight: 8.990633780830471


In [7]:
# -----------------------------
# Inspect weighted FT-Transformer results
# -----------------------------

print("Weighted FT-Transformer — pooled metrics")
print("----------------------------------------")

summary = weighted_ft_cv_result.summary_metrics

print(
    "PR-AUC:",
    f"{summary['pooled_average_precision']:.6f}",
    f"({summary['pooled_average_precision'] * 100:.2f}%)"
)

print(
    "ROC-AUC:",
    f"{summary['pooled_roc_auc']:.6f}",
    f"({summary['pooled_roc_auc'] * 100:.2f}%)"
)

print(
    "F1:",
    f"{summary['pooled_f1']:.6f}",
    f"({summary['pooled_f1'] * 100:.2f}%)"
)

print(
    "Brier Score:",
    f"{summary['pooled_brier_score']:.6f}"
)


print("\n5-fold mean ± SD")
print("----------------")

print(
    "PR-AUC:",
    f"{summary['cv_mean_average_precision']:.6f}",
    "±",
    f"{summary['cv_std_average_precision']:.6f}"
)

print(
    "ROC-AUC:",
    f"{summary['cv_mean_roc_auc']:.6f}",
    "±",
    f"{summary['cv_std_roc_auc']:.6f}"
)

print(
    "F1:",
    f"{summary['cv_mean_f1']:.6f}",
    "±",
    f"{summary['cv_std_f1']:.6f}"
)


print("\nFold-level metrics:")
display(
    weighted_ft_cv_result.fold_metrics
)

Weighted FT-Transformer — pooled metrics
----------------------------------------
PR-AUC: 0.098820 (9.88%)
ROC-AUC: 0.495721 (49.57%)
F1: 0.166441 (16.64%)
Brier Score: 0.251497

5-fold mean ± SD
----------------
PR-AUC: 0.098917 ± 0.002297
ROC-AUC: 0.494650 ± 0.008033
F1: 0.165884 ± 0.005804

Fold-level metrics:


,average_precision,roc_auc,precision,recall,f1,balanced_accuracy,accuracy,brier_score,tn,fp,fn,tp,predicted_positive_count,predicted_positive_rate,fold,fit_time_seconds,predict_time_seconds
0,0.097119,0.489921,0.096327,0.434104,0.157668,0.490647,0.535875,0.247108,7879,6520,906,695,7215,0.450937,1,91.588546,0.422052
1,0.101821,0.504905,0.101658,0.409744,0.162900,0.503573,0.578625,0.247860,8602,5797,945,656,6453,0.403313,2,160.785029,0.231031
2,0.096267,0.483871,0.097312,0.583021,0.166786,0.490635,0.416750,0.255135,5734,8664,668,934,9598,0.599875,3,244.084570,0.251736
3,0.100469,0.496719,0.098975,0.590512,0.169534,0.496187,0.420750,0.250693,5786,8612,656,946,9558,0.597375,4,313.831654,0.222035
4,0.098908,0.497833,0.100844,0.596754,0.172532,0.502364,0.426875,0.256691,5874,8524,646,956,9480,0.592500,5,154.782610,0.455462


## FT-Transformer Experiment Progress and Class-Imbalance Handling

The FT-Transformer experiment was developed and evaluated using the project's frozen 5-fold cross-validation setup on the 80,000-row development partition. The 20,000-row formal holdout set remained excluded throughout model development.

### 1. Initial FT-Transformer Setup

The model used all 17 approved predictors:

- 13 numerical / ordinal features
- 4 categorical features

Numerical features were standardized using `StandardScaler`, while categorical features were encoded independently inside each training fold.

A fresh FT-Transformer model and fresh preprocessing objects were created for every cross-validation fold to prevent information leakage.

The model used the following main architecture:

- 3 Transformer blocks
- Block dimension: 64
- 4 attention heads
- Feed-forward hidden dimension: 128
- Attention dropout: 0.1
- Feed-forward dropout: 0.1
- AdamW optimizer
- 50 training epochs
- Batch size: 512
- Random seed: 42

During implementation, compatibility with the installed `rtdl_revisiting_models` version was also corrected by explicitly supplying:

`ffn_d_hidden_multiplier=None`

The final training environment used an NVIDIA GeForce RTX 4060 Laptop GPU through CUDA-enabled PyTorch.

---

### 2. Initial Unweighted FT-Transformer Experiment

The first FT-Transformer experiment used the standard binary cross-entropy loss:

`BCEWithLogitsLoss()`

No class weighting was applied.

However, the resignation target was strongly imbalanced:

- Approximately 89.99% of employees did not resign.
- Approximately 10.01% of employees resigned.

The initial unweighted model produced the following pooled out-of-fold results:

| Metric | Result |
|---|---:|
| PR-AUC | 10.11% |
| ROC-AUC | 49.88% |
| F1-score at threshold 0.5 | 0.00% |
| Brier Score | 0.0907 |

The F1-score was 0 because the model assigned low resignation probabilities to almost all employees. At the standard classification threshold of 0.5, no useful positive resignation predictions were produced, resulting in zero recall and therefore an F1-score of zero.

More importantly, PR-AUC and ROC-AUC were also close to no-skill performance. Since the resignation prevalence was approximately 10%, a PR-AUC of approximately 10% indicated that the model had very little ability to rank actual resigners above non-resigners.

---

### 3. Introducing Fold-Local Class Weighting

Because the project requires class-imbalance handling, the FT-Transformer training loss was updated to use a positive-class weight.

For each training fold, the positive-class weight was calculated independently as:

`pos_weight = number of negative training samples / number of positive training samples`

The weight was calculated only from the current outer training fold. The validation fold and formal holdout were never used when calculating the class weight.

The weighted loss therefore became:

`BCEWithLogitsLoss(pos_weight=pos_weight_tensor)`

With the approximately 90/10 class distribution, the calculated positive-class weight was close to 9 for each fold.

For example, the final fold produced:

`pos_weight ≈ 8.99`

This means that incorrectly handling an employee who actually resigned contributed substantially more to the training loss than incorrectly handling a non-resigned employee.

A small weighted smoke test was performed before the full experiment. It confirmed that the fold-local weight was calculated correctly and that weighted FT-Transformer training and prediction worked successfully.

---

### 4. Weighted FT-Transformer Experiment

The complete weighted FT-Transformer was then evaluated again using the same frozen 5-fold development CV structure.

Each fold contained:

- 64,000 training employees
- 16,000 validation employees

The full weighted experiment completed successfully on the RTX 4060 GPU in approximately 16.12 minutes.

The weighted model produced:

| Metric | Unweighted | Weighted |
|---|---:|---:|
| PR-AUC | 10.11% | 9.88% |
| ROC-AUC | 49.88% | 49.57% |
| F1-score | 0.00% | 16.64% |
| Brier Score | 0.0907 | 0.2515 |

The weighted 5-fold mean results were:

- PR-AUC: 9.89% ± 0.23%
- ROC-AUC: 49.47% ± 0.80%
- F1-score: 16.59% ± 0.58%

---

### 5. Interpretation

Class weighting clearly changed the model's classification behaviour.

The unweighted model assigned low resignation probabilities to almost everyone, causing the default 0.5 threshold to produce an F1-score of zero.

After applying class weighting, the model became much more willing to classify employees as potential resigners. As a result:

- Recall increased substantially.
- F1-score increased from 0% to approximately 16.64%.
- The model detected many more actual resignation cases.

However, this improvement came with a large number of false-positive predictions.

Across the folds, precision remained close to approximately 10%, while recall was approximately 41% to 60%. Therefore, although the weighted model detected more resigners, many employees predicted as likely to resign were actually non-resigners.

More importantly, class weighting did not improve the model's ranking performance:

- Weighted PR-AUC remained approximately equal to the 10% resignation prevalence.
- Weighted ROC-AUC remained approximately 50%.
- Balanced accuracy also remained close to 50%.

The Brier Score also increased from approximately 0.091 to 0.251, showing that the weighted model's raw probabilities became less well calibrated.

---

### 6. Current Conclusion

Class imbalance was an important reason why the initial FT-Transformer produced an F1-score of zero at the default 0.5 threshold.

Applying fold-local positive-class weighting successfully corrected this behaviour and substantially improved recall and F1-score.

However, class weighting did not improve PR-AUC or ROC-AUC. Both remained approximately at no-skill levels.

Therefore, the results suggest that class imbalance alone is not the main limitation of the current FT-Transformer model. The available predictors may contain limited information for distinguishing employees who resign from employees who remain, or the current model configuration may require further tuning.

The weighted FT-Transformer remains methodologically important because it satisfies the class-imbalance requirement and demonstrates the effect of weighting. The unweighted experiment is retained as a baseline for comparison.

Further analysis should examine the out-of-fold probability distribution and threshold behaviour before making the final FT-Transformer assessment.

In [8]:
# -----------------------------
# Inspect weighted OOF predictions
# -----------------------------

weighted_oof = (
    weighted_ft_cv_result
    .predictions
    .copy()
)


print(
    "OOF prediction shape:",
    weighted_oof.shape
)

print(
    "\nColumns:"
)

print(
    weighted_oof.columns.tolist()
)


print(
    "\nFirst 5 rows:"
)

display(
    weighted_oof.head()
)


print(
    "\nMissing values per column:"
)

print(
    weighted_oof
    .isna()
    .sum()
)

OOF prediction shape: (80000, 6)

Columns:
['Employee_ID', 'cv_fold', 'y_true', 'model_name', 'y_probability', 'y_pred_0_5']

First 5 rows:


,Employee_ID,cv_fold,y_true,model_name,y_probability,y_pred_0_5
0,1,3,0,FT-Transformer,0.498213,0
1,2,3,0,FT-Transformer,0.499968,0
2,4,5,0,FT-Transformer,0.492771,0
3,5,3,0,FT-Transformer,0.508392,1
4,7,3,0,FT-Transformer,0.502569,1



Missing values per column:
Employee_ID      0
cv_fold          0
y_true           0
model_name       0
y_probability    0
y_pred_0_5       0
dtype: int64


In [9]:
# -----------------------------
# Weighted FT-Transformer threshold analysis
# -----------------------------

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)


# Actual labels and OOF probabilities.
y_true_oof = (
    weighted_oof["y_true"]
    .astype(int)
    .to_numpy()
)

y_prob_oof = (
    weighted_oof["y_probability"]
    .astype(float)
    .to_numpy()
)


# Test thresholds from 0.05 to 0.95.
thresholds = np.arange(
    0.05,
    0.951,
    0.01
)


threshold_results = []


for threshold in thresholds:

    y_pred = (
        y_prob_oof >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_oof,
        y_pred,
        labels=[0, 1]
    ).ravel()

    threshold_results.append(
        {
            "threshold": threshold,

            "precision": precision_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "recall": recall_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "f1": f1_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_true_oof,
                y_pred
            ),

            "accuracy": accuracy_score(
                y_true_oof,
                y_pred
            ),

            "predicted_positive_rate": y_pred.mean(),

            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "true_negatives": tn,
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


# --------------------------------------------------
# Find threshold with highest OOF F1
# --------------------------------------------------

best_f1_row = (
    threshold_df
    .loc[
        threshold_df["f1"].idxmax()
    ]
)


print("Best OOF threshold by F1")
print("------------------------")

print(
    "Threshold:",
    f"{best_f1_row['threshold']:.2f}"
)

print(
    "Precision:",
    f"{best_f1_row['precision'] * 100:.2f}%"
)

print(
    "Recall:",
    f"{best_f1_row['recall'] * 100:.2f}%"
)

print(
    "F1:",
    f"{best_f1_row['f1'] * 100:.2f}%"
)

print(
    "Balanced accuracy:",
    f"{best_f1_row['balanced_accuracy'] * 100:.2f}%"
)

print(
    "Predicted positive rate:",
    f"{best_f1_row['predicted_positive_rate'] * 100:.2f}%"
)


print("\nConfusion matrix counts:")

print(
    "TP:",
    int(best_f1_row["true_positives"])
)

print(
    "FP:",
    int(best_f1_row["false_positives"])
)

print(
    "FN:",
    int(best_f1_row["false_negatives"])
)

print(
    "TN:",
    int(best_f1_row["true_negatives"])
)


# --------------------------------------------------
# Compare useful thresholds around 0.50
# --------------------------------------------------

comparison_thresholds = [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
]


display(
    threshold_df[
        threshold_df["threshold"]
        .round(2)
        .isin(comparison_thresholds)
    ][
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "balanced_accuracy",
            "accuracy",
            "predicted_positive_rate",
        ]
    ]
)

Best OOF threshold by F1
------------------------
Threshold: 0.05
Precision: 10.01%
Recall: 100.00%
F1: 18.20%
Balanced accuracy: 50.00%
Predicted positive rate: 100.00%

Confusion matrix counts:
TP: 8008
FP: 71992
FN: 0
TN: 0


,threshold,precision,recall,f1,balanced_accuracy,accuracy,predicted_positive_rate
35,0.40,0.100061,0.983142,0.181636,0.499787,0.113200,0.983525
40,0.45,0.100085,0.908966,0.180316,0.499926,0.172775,0.909100
45,0.50,0.098974,0.522852,0.166441,0.496695,0.475775,0.528800
50,0.55,0.096268,0.062812,0.076022,0.498611,0.847163,0.065312
55,0.60,0.108200,0.011863,0.021382,0.500493,0.891300,0.010975


In [10]:
# -----------------------------
# Save weighted fixed FT-Transformer experiment
# -----------------------------

import json
from pathlib import Path


# --------------------------------------------------
# Output directory
# --------------------------------------------------

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

WEIGHTED_FIXED_DIR = (
    RESULTS_DIR
    / "weighted_fixed_baseline"
)

WEIGHTED_FIXED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# 1. Save OOF predictions
# --------------------------------------------------

weighted_oof_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_oof_predictions.csv"
)

weighted_ft_cv_result.predictions.to_csv(
    weighted_oof_path,
    index=False,
)


# --------------------------------------------------
# 2. Save fold-level metrics
# --------------------------------------------------

weighted_fold_metrics_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_fold_metrics.csv"
)

weighted_ft_cv_result.fold_metrics.to_csv(
    weighted_fold_metrics_path,
    index=False,
)


# --------------------------------------------------
# 3. Save pooled / summary metrics
# --------------------------------------------------

weighted_summary_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_summary_metrics.json"
)

with open(
    weighted_summary_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        weighted_ft_cv_result.summary_metrics,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# 4. Save threshold analysis
# --------------------------------------------------

threshold_analysis_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_threshold_analysis.csv"
)

threshold_df.to_csv(
    threshold_analysis_path,
    index=False,
)


# --------------------------------------------------
# 5. Save run metadata
# --------------------------------------------------

run_metadata = {
    "model": "FT-Transformer",
    "experiment": "weighted_fixed_baseline",

    "development_rows": 80_000,
    "formal_holdout_rows_used": 0,

    "features": 17,
    "cv_folds": 5,

    "epochs_per_fold": 50,
    "batch_size": 512,

    "class_weighting": "fold_local_pos_weight",
    "pos_weight_formula": "negative_count / positive_count",

    "final_fold_pos_weight": (
        weighted_ft_cv_result
        .metadata
        .get("pos_weight")
    ),

    "training_device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),

    "elapsed_minutes": weighted_cv_elapsed / 60,

    "primary_metric": "PR-AUC",

    "pooled_pr_auc": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_average_precision"
        ]
    ),

    "pooled_roc_auc": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_roc_auc"
        ]
    ),

    "pooled_f1_at_0_5": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_f1"
        ]
    ),

    "pooled_brier_score": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_brier_score"
        ]
    ),

    "best_oof_f1_threshold": float(
        best_f1_row["threshold"]
    ),

    "best_oof_f1": float(
        best_f1_row["f1"]
    ),

    "best_threshold_predicted_positive_rate": float(
        best_f1_row[
            "predicted_positive_rate"
        ]
    ),

    "threshold_conclusion": (
        "The numerically best OOF F1 threshold predicts "
        "essentially/all development employees as positive, "
        "so it is not considered a useful decision threshold."
    ),
}


run_metadata_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_run_metadata.json"
)

with open(
    run_metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        run_metadata,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# Verification
# --------------------------------------------------

saved_files = [
    weighted_oof_path,
    weighted_fold_metrics_path,
    weighted_summary_path,
    threshold_analysis_path,
    run_metadata_path,
]


print("Weighted fixed experiment saved.")
print("\nDirectory:")
print(WEIGHTED_FIXED_DIR)

print("\nSaved files:")

for path in saved_files:
    print(
        " ✓",
        path.name,
        f"({path.stat().st_size:,} bytes)"
    )


# Critical integrity checks
assert len(
    weighted_ft_cv_result.predictions
) == 80_000

assert (
    weighted_ft_cv_result
    .predictions["Employee_ID"]
    .is_unique
)

assert set(
    weighted_ft_cv_result
    .predictions["Employee_ID"]
).isdisjoint(
    holdout_ids
)

assert all(
    path.exists()
    for path in saved_files
)


print(
    "\nWeighted experiment artifact checks PASSED."
)

Weighted fixed experiment saved.

Directory:
D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\weighted_fixed_baseline

Saved files:
 ✓ ft_transformer_weighted_oof_predictions.csv (3,760,368 bytes)
 ✓ ft_transformer_weighted_fold_metrics.csv (1,282 bytes)
 ✓ ft_transformer_weighted_summary_metrics.json (708 bytes)
 ✓ ft_transformer_weighted_threshold_analysis.csv (9,251 bytes)
 ✓ ft_transformer_weighted_run_metadata.json (1,034 bytes)

Weighted experiment artifact checks PASSED.


In [11]:
# -----------------------------
# Early-stopping FT-Transformer smoke test
# -----------------------------

import importlib
import numpy as np

import src.models.ft_transformer as ft_module


# Reload the updated source file.
ft_module = importlib.reload(ft_module)
FTTransformerAdapter = ft_module.FTTransformerAdapter


# --------------------------------------------------
# Small development-only sample
# --------------------------------------------------

smoke_df = (
    development_df
    .sample(
        n=4096,
        random_state=42
    )
    .reset_index(drop=True)
)

X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = (
    smoke_df[target_column]
    .astype(int)
    .copy()
)


# --------------------------------------------------
# Short smoke-test configuration
# --------------------------------------------------
#
# These shorter settings are ONLY for checking that
# inner validation, weighting, scheduler and best-model
# restoration work correctly.
#
# They are NOT the final experiment settings.

smoke_params = fixed_params.copy()

smoke_params.update(
    {
        "max_epochs": 8,
        "patience": 3,
        "validation_fraction_inner": 0.15,
        "scheduler_factor": 0.5,
        "scheduler_patience": 2,
        "min_delta": 1e-6,
    }
)


# --------------------------------------------------
# Create and fit adapter
# --------------------------------------------------

early_stop_smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=smoke_params,
    categorical_features=categorical_feature_names.copy(),
)


early_stop_smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=None,
    batch_size=256,
)


metadata = (
    early_stop_smoke_adapter
    .get_metadata()
)


# --------------------------------------------------
# Check predictions
# --------------------------------------------------

smoke_probs = (
    early_stop_smoke_adapter
    .predict_proba(
        X_smoke.iloc[:128]
    )
)


# --------------------------------------------------
# Display training diagnostics
# --------------------------------------------------

print("Early-stopping smoke test")
print("-------------------------")

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "\nOuter sample size:",
    len(X_smoke)
)

print(
    "Inner training size:",
    metadata["inner_train_size"]
)

print(
    "Inner validation size:",
    metadata["inner_validation_size"]
)

print(
    "Fold-local pos_weight:",
    metadata["pos_weight"]
)

print(
    "Best epoch:",
    metadata["best_epoch"]
)

print(
    "Best inner PR-AUC:",
    metadata["best_inner_pr_auc"]
)

print(
    "Epochs actually trained:",
    metadata["epochs_trained"]
)

print(
    "Class weighting:",
    metadata["class_weighting"]
)


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert metadata["pos_weight"] is not None
assert metadata["pos_weight"] > 1

assert metadata["best_epoch"] is not None
assert metadata["best_epoch"] >= 1

assert metadata["best_inner_pr_auc"] is not None

assert 0 < metadata["best_inner_pr_auc"] <= 1

assert (
    metadata["epochs_trained"]
    <= smoke_params["max_epochs"]
)

assert (
    metadata["inner_train_size"]
    + metadata["inner_validation_size"]
    == len(X_smoke)
)

assert len(smoke_probs) == 128

assert np.isfinite(
    smoke_probs
).all()

assert (
    (smoke_probs >= 0)
    & (smoke_probs <= 1)
).all()


print(
    "\nEarly-stopping FT-Transformer smoke test PASSED."
)

print(
    "Probability range:",
    f"{smoke_probs.min():.6f}",
    "to",
    f"{smoke_probs.max():.6f}"
)

Early-stopping smoke test
-------------------------
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Outer sample size: 4096
Inner training size: 3481
Inner validation size: 615
Fold-local pos_weight: 9.71076923076923
Best epoch: 6
Best inner PR-AUC: 0.09358474637174255
Epochs actually trained: 8
Class weighting: inner_training_fold_positive_class_weight

Early-stopping FT-Transformer smoke test PASSED.
Probability range: 0.411608 to 0.552710


In [12]:
# -----------------------------
# Single full-fold early-stopping test
# -----------------------------

import time

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    brier_score_loss,
)


# --------------------------------------------------
# Select one frozen outer fold
# --------------------------------------------------

TEST_FOLD = 1

outer_train_df = (
    development_df.loc[
        development_df["cv_fold"] != TEST_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)

outer_val_df = (
    development_df.loc[
        development_df["cv_fold"] == TEST_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(outer_train_df) == 64_000
assert len(outer_val_df) == 16_000


X_outer_train = outer_train_df[
    primary_model_features
].copy()

y_outer_train = (
    outer_train_df[target_column]
    .astype(int)
    .copy()
)

X_outer_val = outer_val_df[
    primary_model_features
].copy()

y_outer_val = (
    outer_val_df[target_column]
    .astype(int)
    .copy()
)


# --------------------------------------------------
# Full early-stopping configuration
# --------------------------------------------------

single_fold_params = fixed_params.copy()

single_fold_params.update(
    {
        "max_epochs": 100,
        "patience": 15,
        "validation_fraction_inner": 0.15,
        "scheduler_factor": 0.5,
        "scheduler_patience": 5,
        "min_delta": 1e-6,
    }
)


# --------------------------------------------------
# Create fresh model
# --------------------------------------------------

single_fold_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=single_fold_params,
    categorical_features=categorical_feature_names.copy(),
)


print("Full Fold 1 early-stopping test")
print("--------------------------------")
print("Outer training rows:", len(X_outer_train))
print("Outer validation rows:", len(X_outer_val))

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("\nTraining...")


# --------------------------------------------------
# Train
# --------------------------------------------------

start_time = time.perf_counter()

single_fold_adapter.fit(
    X=X_outer_train,
    y=y_outer_train,
    epochs=None,
    batch_size=512,
)

elapsed_seconds = (
    time.perf_counter()
    - start_time
)


# --------------------------------------------------
# Predict untouched outer validation fold
# --------------------------------------------------

val_prob = single_fold_adapter.predict_proba(
    X_outer_val
)

val_pred = (
    val_prob >= 0.5
).astype(int)


# --------------------------------------------------
# Metrics
# --------------------------------------------------

fold_pr_auc = average_precision_score(
    y_outer_val,
    val_prob,
)

fold_roc_auc = roc_auc_score(
    y_outer_val,
    val_prob,
)

fold_precision = precision_score(
    y_outer_val,
    val_pred,
    zero_division=0,
)

fold_recall = recall_score(
    y_outer_val,
    val_pred,
    zero_division=0,
)

fold_f1 = f1_score(
    y_outer_val,
    val_pred,
    zero_division=0,
)

fold_balanced_accuracy = balanced_accuracy_score(
    y_outer_val,
    val_pred,
)

fold_brier = brier_score_loss(
    y_outer_val,
    val_prob,
)


metadata = single_fold_adapter.get_metadata()


# --------------------------------------------------
# Results
# --------------------------------------------------

print("\nTraining diagnostics")
print("--------------------")

print(
    "Inner training rows:",
    metadata["inner_train_size"]
)

print(
    "Inner validation rows:",
    metadata["inner_validation_size"]
)

print(
    "pos_weight:",
    f"{metadata['pos_weight']:.4f}"
)

print(
    "Best epoch:",
    metadata["best_epoch"]
)

print(
    "Epochs trained:",
    metadata["epochs_trained"]
)

print(
    "Best inner PR-AUC:",
    f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
)

print(
    "Training time:",
    f"{elapsed_seconds / 60:.2f} minutes"
)


print("\nUntouched outer Fold 1 results")
print("------------------------------")

print(
    "PR-AUC:",
    f"{fold_pr_auc * 100:.2f}%"
)

print(
    "ROC-AUC:",
    f"{fold_roc_auc * 100:.2f}%"
)

print(
    "Precision:",
    f"{fold_precision * 100:.2f}%"
)

print(
    "Recall:",
    f"{fold_recall * 100:.2f}%"
)

print(
    "F1:",
    f"{fold_f1 * 100:.2f}%"
)

print(
    "Balanced accuracy:",
    f"{fold_balanced_accuracy * 100:.2f}%"
)

print(
    "Brier score:",
    f"{fold_brier:.6f}"
)

print(
    "Probability range:",
    f"{val_prob.min():.4f}",
    "to",
    f"{val_prob.max():.4f}"
)

Full Fold 1 early-stopping test
--------------------------------
Outer training rows: 64000
Outer validation rows: 16000
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Training...

Training diagnostics
--------------------
Inner training rows: 54400
Inner validation rows: 9600
pos_weight: 8.9890
Best epoch: 22
Epochs trained: 37
Best inner PR-AUC: 10.76%
Training time: 1.12 minutes

Untouched outer Fold 1 results
------------------------------
PR-AUC: 9.66%
ROC-AUC: 48.59%
Precision: 9.57%
Recall: 34.23%
F1: 14.96%
Balanced accuracy: 49.14%
Brier score: 0.242199
Probability range: 0.2983 to 0.6549


In [13]:
# -----------------------------
# FT-Transformer learning-rate screening
# -----------------------------
#
# IMPORTANT:
# Hyperparameters are compared using ONLY the inner
# validation split created from the 64,000-row outer
# training fold.
#
# The untouched 16,000-row outer Fold 1 validation
# set is NOT evaluated in this cell.
# -----------------------------

import time
import pandas as pd


learning_rates = [
    0.0001,
    0.0003,
    0.0010,
]


lr_screen_results = []


for learning_rate in learning_rates:

    print(
        f"\nTesting learning rate: {learning_rate}"
    )

    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            "learning_rate": learning_rate,

            "max_epochs": 100,
            "patience": 15,

            "validation_fraction_inner": 0.15,

            "scheduler_factor": 0.5,
            "scheduler_patience": 5,

            "min_delta": 1e-6,
        }
    )


    # Fresh model for every candidate.
    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    lr_screen_results.append(
        {
            "learning_rate": learning_rate,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

lr_screen_df = pd.DataFrame(
    lr_screen_results
)

lr_screen_df = (
    lr_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nLearning-rate screening results"
)

display(
    lr_screen_df
)


best_lr = float(
    lr_screen_df.iloc[0][
        "learning_rate"
    ]
)


print(
    "\nBest learning rate by inner-validation PR-AUC:",
    best_lr
)

print(
    "Best inner-validation PR-AUC:",
    f"{lr_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing learning rate: 0.0001
  Best inner PR-AUC: 10.88%
  Best epoch: 3
  Epochs trained: 18
  Time: 0.44 min

Testing learning rate: 0.0003
  Best inner PR-AUC: 10.73%
  Best epoch: 32
  Epochs trained: 47
  Time: 1.21 min

Testing learning rate: 0.001
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 0.96 min

Learning-rate screening results


,learning_rate,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,0.0001,0.108789,3,18,8.988983,0.438182
1,0.0010,0.107563,22,37,8.988983,0.962459
2,0.0003,0.107348,32,47,8.988983,1.211724



Best learning rate by inner-validation PR-AUC: 0.0001
Best inner-validation PR-AUC: 10.88%


In [14]:
# -----------------------------
# Learning-rate robustness check
# -----------------------------
#
# Compare the two strongest learning rates across
# several random seeds.
#
# Hyperparameter selection still uses ONLY inner
# validation data from the outer-training partition.
# -----------------------------

import time
import pandas as pd


candidate_learning_rates = [
    0.0001,
    0.0010,
]

robustness_seeds = [
    42,
    123,
    2026,
]


robustness_results = []


for learning_rate in candidate_learning_rates:

    for seed in robustness_seeds:

        print(
            f"\nLR={learning_rate} | Seed={seed}"
        )


        candidate_params = fixed_params.copy()

        candidate_params.update(
            {
                "learning_rate": learning_rate,
                "seed": seed,

                "max_epochs": 100,
                "patience": 15,

                "validation_fraction_inner": 0.15,

                "scheduler_factor": 0.5,
                "scheduler_patience": 5,

                "min_delta": 1e-6,
            }
        )


        candidate_adapter = FTTransformerAdapter(
            name=model_name,
            model_id=model_id,
            params=candidate_params,
            categorical_features=(
                categorical_feature_names.copy()
            ),
        )


        start_time = time.perf_counter()


        candidate_adapter.fit(
            X=X_outer_train,
            y=y_outer_train,
            epochs=None,
            batch_size=512,
        )


        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )


        metadata = (
            candidate_adapter
            .get_metadata()
        )


        robustness_results.append(
            {
                "learning_rate": learning_rate,
                "seed": seed,

                "best_inner_pr_auc": (
                    metadata[
                        "best_inner_pr_auc"
                    ]
                ),

                "best_epoch": (
                    metadata[
                        "best_epoch"
                    ]
                ),

                "epochs_trained": (
                    metadata[
                        "epochs_trained"
                    ]
                ),

                "training_minutes": (
                    elapsed_seconds / 60
                ),
            }
        )


        print(
            "  Best inner PR-AUC:",
            f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
        )

        print(
            "  Best epoch:",
            metadata["best_epoch"]
        )

        print(
            "  Epochs trained:",
            metadata["epochs_trained"]
        )


# --------------------------------------------------
# Detailed results
# --------------------------------------------------

lr_robustness_df = pd.DataFrame(
    robustness_results
)


print(
    "\nDetailed robustness results"
)

display(
    lr_robustness_df
)


# --------------------------------------------------
# Aggregate by learning rate
# --------------------------------------------------

lr_robustness_summary = (
    lr_robustness_df
    .groupby(
        "learning_rate",
        as_index=False,
    )
    .agg(
        mean_inner_pr_auc=(
            "best_inner_pr_auc",
            "mean",
        ),

        std_inner_pr_auc=(
            "best_inner_pr_auc",
            "std",
        ),

        min_inner_pr_auc=(
            "best_inner_pr_auc",
            "min",
        ),

        max_inner_pr_auc=(
            "best_inner_pr_auc",
            "max",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .sort_values(
        "mean_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nLearning-rate robustness summary"
)

display(
    lr_robustness_summary
)


robust_best_lr = float(
    lr_robustness_summary.iloc[0][
        "learning_rate"
    ]
)


print(
    "\nRobust best learning rate:",
    robust_best_lr
)

print(
    "Mean inner PR-AUC:",
    f"{lr_robustness_summary.iloc[0]['mean_inner_pr_auc'] * 100:.2f}%"
)

print(
    "SD:",
    f"{lr_robustness_summary.iloc[0]['std_inner_pr_auc'] * 100:.2f}%"
)


LR=0.0001 | Seed=42
  Best inner PR-AUC: 10.88%
  Best epoch: 3
  Epochs trained: 18

LR=0.0001 | Seed=123
  Best inner PR-AUC: 10.55%
  Best epoch: 3
  Epochs trained: 18

LR=0.0001 | Seed=2026
  Best inner PR-AUC: 10.30%
  Best epoch: 11
  Epochs trained: 26

LR=0.001 | Seed=42
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37

LR=0.001 | Seed=123
  Best inner PR-AUC: 10.76%
  Best epoch: 3
  Epochs trained: 18

LR=0.001 | Seed=2026
  Best inner PR-AUC: 10.46%
  Best epoch: 6
  Epochs trained: 21

Detailed robustness results


,learning_rate,seed,best_inner_pr_auc,best_epoch,epochs_trained,training_minutes
0,0.0001,42,0.108789,3,18,0.485011
1,0.0001,123,0.105475,3,18,0.472696
2,0.0001,2026,0.102958,11,26,1.169585
3,0.0010,42,0.107563,22,37,0.956619
4,0.0010,123,0.107627,3,18,0.589365
5,0.0010,2026,0.104620,6,21,1.184796



Learning-rate robustness summary


,learning_rate,mean_inner_pr_auc,std_inner_pr_auc,min_inner_pr_auc,max_inner_pr_auc,mean_best_epoch
0,0.0010,0.106603,0.001718,0.104620,0.107627,10.333333
1,0.0001,0.105740,0.002924,0.102958,0.108789,5.666667



Robust best learning rate: 0.001
Mean inner PR-AUC: 10.66%
SD: 0.17%


In [15]:
# -----------------------------
# FT-Transformer d_block screening
# -----------------------------
#
# Change ONE architecture variable only:
# transformer block width (d_block).
#
# Selection uses ONLY inner-validation PR-AUC.
# The 16,000-row outer Fold 1 validation set
# remains untouched.
# -----------------------------

import time
import pandas as pd


d_block_values = [
    32,
    64,
    128,
]


d_block_results = []


for d_block in d_block_values:

    print(
        f"\nTesting d_block: {d_block}"
    )


    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            # Current robust learning-rate choice
            "learning_rate": 0.001,

            # Variable being tested
            "d_block": d_block,

            # Keep architecture otherwise unchanged
            "n_blocks": 3,
            "attention_n_heads": 4,
            "ffn_d_hidden": 128,

            # Training controls
            "seed": 42,
            "max_epochs": 100,
            "patience": 15,
            "validation_fraction_inner": 0.15,
            "scheduler_factor": 0.5,
            "scheduler_patience": 5,
            "min_delta": 1e-6,
        }
    )


    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    d_block_results.append(
        {
            "d_block": d_block,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

d_block_screen_df = pd.DataFrame(
    d_block_results
)

d_block_screen_df = (
    d_block_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nd_block screening results"
)

display(
    d_block_screen_df
)


best_d_block = int(
    d_block_screen_df.iloc[0][
        "d_block"
    ]
)


print(
    "\nBest d_block by inner-validation PR-AUC:",
    best_d_block
)

print(
    "Best inner-validation PR-AUC:",
    f"{d_block_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing d_block: 32
  Best inner PR-AUC: 10.56%
  Best epoch: 8
  Epochs trained: 23
  Time: 0.59 min

Testing d_block: 64
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 2.03 min

Testing d_block: 128
  Best inner PR-AUC: 10.55%
  Best epoch: 8
  Epochs trained: 23
  Time: 0.86 min

d_block screening results


,d_block,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,64,0.107563,22,37,8.988983,2.031940
1,32,0.105603,8,23,8.988983,0.589102
2,128,0.105526,8,23,8.988983,0.856013



Best d_block by inner-validation PR-AUC: 64
Best inner-validation PR-AUC: 10.76%


In [16]:
# -----------------------------
# FT-Transformer n_blocks screening
# -----------------------------
#
# Change ONE architecture variable only:
# number of Transformer blocks.
#
# Selection uses ONLY inner-validation PR-AUC.
# Outer Fold 1 validation remains untouched.
# -----------------------------

import time
import pandas as pd


n_block_values = [
    2,
    3,
    4,
]


n_block_results = []


for n_blocks in n_block_values:

    print(
        f"\nTesting n_blocks: {n_blocks}"
    )


    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            # Choices retained from previous tests
            "learning_rate": 0.001,
            "d_block": 64,

            # Variable being tested
            "n_blocks": n_blocks,

            # Keep remaining architecture fixed
            "attention_n_heads": 4,
            "ffn_d_hidden": 128,

            # Training controls
            "seed": 42,
            "max_epochs": 100,
            "patience": 15,
            "validation_fraction_inner": 0.15,
            "scheduler_factor": 0.5,
            "scheduler_patience": 5,
            "min_delta": 1e-6,
        }
    )


    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    n_block_results.append(
        {
            "n_blocks": n_blocks,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

n_block_screen_df = pd.DataFrame(
    n_block_results
)

n_block_screen_df = (
    n_block_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nn_blocks screening results"
)

display(
    n_block_screen_df
)


best_n_blocks = int(
    n_block_screen_df.iloc[0][
        "n_blocks"
    ]
)


print(
    "\nBest n_blocks by inner-validation PR-AUC:",
    best_n_blocks
)

print(
    "Best inner-validation PR-AUC:",
    f"{n_block_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing n_blocks: 2
  Best inner PR-AUC: 10.84%
  Best epoch: 10
  Epochs trained: 25
  Time: 0.58 min

Testing n_blocks: 3
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 1.00 min

Testing n_blocks: 4
  Best inner PR-AUC: 10.50%
  Best epoch: 6
  Epochs trained: 21
  Time: 0.62 min

n_blocks screening results


,n_blocks,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,2,0.108444,10,25,8.988983,0.584676
1,3,0.107563,22,37,8.988983,1.004432
2,4,0.105000,6,21,8.988983,0.616395



Best n_blocks by inner-validation PR-AUC: 2
Best inner-validation PR-AUC: 10.84%


In [17]:
# -----------------------------
# n_blocks robustness check
# -----------------------------
#
# Compare the two strongest architectures:
# 2 blocks vs 3 blocks
#
# Selection still uses ONLY inner-validation PR-AUC.
# Outer Fold 1 validation remains untouched.
# -----------------------------

import time
import pandas as pd


candidate_n_blocks = [
    2,
    3,
]

robustness_seeds = [
    42,
    123,
    2026,
]


n_block_robustness_results = []


for n_blocks in candidate_n_blocks:

    for seed in robustness_seeds:

        print(
            f"\nn_blocks={n_blocks} | Seed={seed}"
        )


        candidate_params = fixed_params.copy()

        candidate_params.update(
            {
                "learning_rate": 0.001,
                "d_block": 64,

                "n_blocks": n_blocks,

                "attention_n_heads": 4,
                "ffn_d_hidden": 128,

                "seed": seed,

                "max_epochs": 100,
                "patience": 15,

                "validation_fraction_inner": 0.15,

                "scheduler_factor": 0.5,
                "scheduler_patience": 5,

                "min_delta": 1e-6,
            }
        )


        candidate_adapter = FTTransformerAdapter(
            name=model_name,
            model_id=model_id,
            params=candidate_params,
            categorical_features=(
                categorical_feature_names.copy()
            ),
        )


        start_time = time.perf_counter()


        candidate_adapter.fit(
            X=X_outer_train,
            y=y_outer_train,
            epochs=None,
            batch_size=512,
        )


        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )


        metadata = (
            candidate_adapter
            .get_metadata()
        )


        n_block_robustness_results.append(
            {
                "n_blocks": n_blocks,
                "seed": seed,

                "best_inner_pr_auc": (
                    metadata[
                        "best_inner_pr_auc"
                    ]
                ),

                "best_epoch": (
                    metadata[
                        "best_epoch"
                    ]
                ),

                "epochs_trained": (
                    metadata[
                        "epochs_trained"
                    ]
                ),

                "training_minutes": (
                    elapsed_seconds / 60
                ),
            }
        )


        print(
            "  Best inner PR-AUC:",
            f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
        )

        print(
            "  Best epoch:",
            metadata["best_epoch"]
        )

        print(
            "  Epochs trained:",
            metadata["epochs_trained"]
        )


# --------------------------------------------------
# Detailed results
# --------------------------------------------------

n_block_robustness_df = pd.DataFrame(
    n_block_robustness_results
)


print(
    "\nDetailed n_blocks robustness results"
)

display(
    n_block_robustness_df
)


# --------------------------------------------------
# Aggregate results
# --------------------------------------------------

n_block_robustness_summary = (
    n_block_robustness_df
    .groupby(
        "n_blocks",
        as_index=False,
    )
    .agg(
        mean_inner_pr_auc=(
            "best_inner_pr_auc",
            "mean",
        ),

        std_inner_pr_auc=(
            "best_inner_pr_auc",
            "std",
        ),

        min_inner_pr_auc=(
            "best_inner_pr_auc",
            "min",
        ),

        max_inner_pr_auc=(
            "best_inner_pr_auc",
            "max",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .sort_values(
        "mean_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nn_blocks robustness summary"
)

display(
    n_block_robustness_summary
)


robust_best_n_blocks = int(
    n_block_robustness_summary.iloc[0][
        "n_blocks"
    ]
)


print(
    "\nRobust best n_blocks:",
    robust_best_n_blocks
)

print(
    "Mean inner PR-AUC:",
    f"{n_block_robustness_summary.iloc[0]['mean_inner_pr_auc'] * 100:.2f}%"
)

print(
    "SD:",
    f"{n_block_robustness_summary.iloc[0]['std_inner_pr_auc'] * 100:.2f}%"
)


n_blocks=2 | Seed=42
  Best inner PR-AUC: 10.84%
  Best epoch: 10
  Epochs trained: 25

n_blocks=2 | Seed=123
  Best inner PR-AUC: 10.21%
  Best epoch: 3
  Epochs trained: 18

n_blocks=2 | Seed=2026
  Best inner PR-AUC: 10.44%
  Best epoch: 11
  Epochs trained: 26

n_blocks=3 | Seed=42
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37

n_blocks=3 | Seed=123
  Best inner PR-AUC: 10.76%
  Best epoch: 3
  Epochs trained: 18

n_blocks=3 | Seed=2026
  Best inner PR-AUC: 10.46%
  Best epoch: 6
  Epochs trained: 21

Detailed n_blocks robustness results


,n_blocks,seed,best_inner_pr_auc,best_epoch,epochs_trained,training_minutes
0,2,42,0.108444,10,25,1.751576
1,2,123,0.102136,3,18,1.455392
2,2,2026,0.104428,11,26,1.408384
3,3,42,0.107563,22,37,2.257028
4,3,123,0.107627,3,18,1.855569
5,3,2026,0.104620,6,21,2.062979



n_blocks robustness summary


,n_blocks,mean_inner_pr_auc,std_inner_pr_auc,min_inner_pr_auc,max_inner_pr_auc,mean_best_epoch
0,3,0.106603,0.001718,0.104620,0.107627,10.333333
1,2,0.105002,0.003193,0.102136,0.108444,8.000000



Robust best n_blocks: 3
Mean inner PR-AUC: 10.66%
SD: 0.17%


In [18]:
# -----------------------------
# FT-Transformer FFN hidden-size screening
# -----------------------------
#
# Change ONE architecture variable only:
# feed-forward network hidden dimension.
#
# Selection uses ONLY inner-validation PR-AUC.
# No outer validation predictions are generated here.
# -----------------------------

import time
import pandas as pd


ffn_hidden_values = [
    64,
    128,
    256,
]


ffn_hidden_results = []


for ffn_d_hidden in ffn_hidden_values:

    print(
        f"\nTesting ffn_d_hidden: {ffn_d_hidden}"
    )


    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            # Retained settings
            "learning_rate": 0.001,
            "d_block": 64,
            "n_blocks": 3,

            # Keep attention unchanged
            "attention_n_heads": 4,

            # Variable being tested
            "ffn_d_hidden": ffn_d_hidden,

            # Reproducibility
            "seed": 42,

            # Training controls
            "max_epochs": 100,
            "patience": 15,

            "validation_fraction_inner": 0.15,

            "scheduler_factor": 0.5,
            "scheduler_patience": 5,

            "min_delta": 1e-6,
        }
    )


    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    ffn_hidden_results.append(
        {
            "ffn_d_hidden": ffn_d_hidden,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

ffn_hidden_screen_df = pd.DataFrame(
    ffn_hidden_results
)

ffn_hidden_screen_df = (
    ffn_hidden_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nFFN hidden-size screening results"
)

display(
    ffn_hidden_screen_df
)


best_ffn_hidden = int(
    ffn_hidden_screen_df.iloc[0][
        "ffn_d_hidden"
    ]
)


print(
    "\nBest ffn_d_hidden by inner-validation PR-AUC:",
    best_ffn_hidden
)

print(
    "Best inner-validation PR-AUC:",
    f"{ffn_hidden_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing ffn_d_hidden: 64
  Best inner PR-AUC: 10.95%
  Best epoch: 49
  Epochs trained: 64
  Time: 1.63 min

Testing ffn_d_hidden: 128
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 0.89 min

Testing ffn_d_hidden: 256
  Best inner PR-AUC: 10.55%
  Best epoch: 17
  Epochs trained: 32
  Time: 0.79 min

FFN hidden-size screening results


,ffn_d_hidden,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,64,0.109485,49,64,8.988983,1.628857
1,128,0.107563,22,37,8.988983,0.891710
2,256,0.105534,17,32,8.988983,0.785795



Best ffn_d_hidden by inner-validation PR-AUC: 64
Best inner-validation PR-AUC: 10.95%


In [19]:
# -----------------------------
# FFN hidden-size robustness check
# -----------------------------
#
# Compare ffn_d_hidden = 64 vs 128
# across multiple seeds.
#
# Selection uses ONLY inner-validation PR-AUC.
# Outer Fold 1 validation remains untouched.
# -----------------------------

import time
import pandas as pd


candidate_ffn_hidden = [
    64,
    128,
]

robustness_seeds = [
    42,
    123,
    2026,
]


ffn_robustness_results = []


for ffn_d_hidden in candidate_ffn_hidden:

    for seed in robustness_seeds:

        print(
            f"\nffn_d_hidden={ffn_d_hidden} | Seed={seed}"
        )


        candidate_params = fixed_params.copy()

        candidate_params.update(
            {
                # Retained choices
                "learning_rate": 0.001,
                "d_block": 64,
                "n_blocks": 3,
                "attention_n_heads": 4,

                # Variable being compared
                "ffn_d_hidden": ffn_d_hidden,

                # Reproducibility
                "seed": seed,

                # Training controls
                "max_epochs": 100,
                "patience": 15,

                "validation_fraction_inner": 0.15,

                "scheduler_factor": 0.5,
                "scheduler_patience": 5,

                "min_delta": 1e-6,
            }
        )


        candidate_adapter = FTTransformerAdapter(
            name=model_name,
            model_id=model_id,
            params=candidate_params,
            categorical_features=(
                categorical_feature_names.copy()
            ),
        )


        start_time = time.perf_counter()


        candidate_adapter.fit(
            X=X_outer_train,
            y=y_outer_train,
            epochs=None,
            batch_size=512,
        )


        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )


        metadata = (
            candidate_adapter
            .get_metadata()
        )


        ffn_robustness_results.append(
            {
                "ffn_d_hidden": ffn_d_hidden,
                "seed": seed,

                "best_inner_pr_auc": (
                    metadata[
                        "best_inner_pr_auc"
                    ]
                ),

                "best_epoch": (
                    metadata[
                        "best_epoch"
                    ]
                ),

                "epochs_trained": (
                    metadata[
                        "epochs_trained"
                    ]
                ),

                "training_minutes": (
                    elapsed_seconds / 60
                ),
            }
        )


        print(
            "  Best inner PR-AUC:",
            f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
        )

        print(
            "  Best epoch:",
            metadata["best_epoch"]
        )

        print(
            "  Epochs trained:",
            metadata["epochs_trained"]
        )


# --------------------------------------------------
# Detailed results
# --------------------------------------------------

ffn_robustness_df = pd.DataFrame(
    ffn_robustness_results
)


print(
    "\nDetailed FFN robustness results"
)

display(
    ffn_robustness_df
)


# --------------------------------------------------
# Aggregate results
# --------------------------------------------------

ffn_robustness_summary = (
    ffn_robustness_df
    .groupby(
        "ffn_d_hidden",
        as_index=False,
    )
    .agg(
        mean_inner_pr_auc=(
            "best_inner_pr_auc",
            "mean",
        ),

        std_inner_pr_auc=(
            "best_inner_pr_auc",
            "std",
        ),

        min_inner_pr_auc=(
            "best_inner_pr_auc",
            "min",
        ),

        max_inner_pr_auc=(
            "best_inner_pr_auc",
            "max",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .sort_values(
        "mean_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nFFN robustness summary"
)

display(
    ffn_robustness_summary
)


robust_best_ffn_hidden = int(
    ffn_robustness_summary.iloc[0][
        "ffn_d_hidden"
    ]
)


print(
    "\nRobust best ffn_d_hidden:",
    robust_best_ffn_hidden
)

print(
    "Mean inner PR-AUC:",
    f"{ffn_robustness_summary.iloc[0]['mean_inner_pr_auc'] * 100:.2f}%"
)

print(
    "SD:",
    f"{ffn_robustness_summary.iloc[0]['std_inner_pr_auc'] * 100:.2f}%"
)


ffn_d_hidden=64 | Seed=42
  Best inner PR-AUC: 10.95%
  Best epoch: 49
  Epochs trained: 64

ffn_d_hidden=64 | Seed=123
  Best inner PR-AUC: 10.25%
  Best epoch: 5
  Epochs trained: 20

ffn_d_hidden=64 | Seed=2026
  Best inner PR-AUC: 10.39%
  Best epoch: 12
  Epochs trained: 27

ffn_d_hidden=128 | Seed=42
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37

ffn_d_hidden=128 | Seed=123
  Best inner PR-AUC: 10.76%
  Best epoch: 3
  Epochs trained: 18

ffn_d_hidden=128 | Seed=2026
  Best inner PR-AUC: 10.46%
  Best epoch: 6
  Epochs trained: 21

Detailed FFN robustness results


,ffn_d_hidden,seed,best_inner_pr_auc,best_epoch,epochs_trained,training_minutes
0,64,42,0.109485,49,64,1.548134
1,64,123,0.102515,5,20,0.503817
2,64,2026,0.103917,12,27,0.718128
3,128,42,0.107563,22,37,0.937638
4,128,123,0.107627,3,18,0.442898
5,128,2026,0.104620,6,21,0.508107



FFN robustness summary


,ffn_d_hidden,mean_inner_pr_auc,std_inner_pr_auc,min_inner_pr_auc,max_inner_pr_auc,mean_best_epoch
0,128,0.106603,0.001718,0.104620,0.107627,10.333333
1,64,0.105306,0.003687,0.102515,0.109485,22.000000



Robust best ffn_d_hidden: 128
Mean inner PR-AUC: 10.66%
SD: 0.17%


In [20]:
# -----------------------------
# FT-Transformer attention-head screening
# -----------------------------
#
# Change ONE architecture variable only:
# number of attention heads.
#
# Selection uses ONLY inner-validation PR-AUC.
# The outer Fold 1 validation set is NOT evaluated.
# -----------------------------

import time
import pandas as pd


attention_head_values = [
    2,
    4,
    8,
]


attention_head_results = []


for attention_n_heads in attention_head_values:

    print(
        f"\nTesting attention_n_heads: {attention_n_heads}"
    )


    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            # Retained settings
            "learning_rate": 0.001,
            "d_block": 64,
            "n_blocks": 3,
            "ffn_d_hidden": 128,

            # Variable being tested
            "attention_n_heads": attention_n_heads,

            # Reproducibility
            "seed": 42,

            # Training controls
            "max_epochs": 100,
            "patience": 15,
            "validation_fraction_inner": 0.15,
            "scheduler_factor": 0.5,
            "scheduler_patience": 5,
            "min_delta": 1e-6,
        }
    )


    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    attention_head_results.append(
        {
            "attention_n_heads": attention_n_heads,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

attention_head_screen_df = pd.DataFrame(
    attention_head_results
)

attention_head_screen_df = (
    attention_head_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nAttention-head screening results"
)

display(
    attention_head_screen_df
)


best_attention_heads = int(
    attention_head_screen_df.iloc[0][
        "attention_n_heads"
    ]
)


print(
    "\nBest attention_n_heads by inner-validation PR-AUC:",
    best_attention_heads
)

print(
    "Best inner-validation PR-AUC:",
    f"{attention_head_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing attention_n_heads: 2
  Best inner PR-AUC: 10.72%
  Best epoch: 1
  Epochs trained: 16
  Time: 0.40 min

Testing attention_n_heads: 4
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 0.90 min

Testing attention_n_heads: 8
  Best inner PR-AUC: 10.69%
  Best epoch: 3
  Epochs trained: 18
  Time: 0.47 min

Attention-head screening results


,attention_n_heads,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,4,0.107563,22,37,8.988983,0.904395
1,2,0.107246,1,16,8.988983,0.395126
2,8,0.106862,3,18,8.988983,0.469466



Best attention_n_heads by inner-validation PR-AUC: 4
Best inner-validation PR-AUC: 10.76%


In [21]:
# -----------------------------
# FT-Transformer FFN dropout screening
# -----------------------------
#
# Change ONE regularization variable only:
# feed-forward network dropout.
#
# Selection uses ONLY inner-validation PR-AUC.
# Outer Fold 1 validation remains untouched.
# -----------------------------

import time
import pandas as pd


ffn_dropout_values = [
    0.0,
    0.1,
    0.2,
]


ffn_dropout_results = []


for ffn_dropout in ffn_dropout_values:

    print(
        f"\nTesting ffn_dropout: {ffn_dropout}"
    )


    candidate_params = fixed_params.copy()

    candidate_params.update(
        {
            # Retained architecture
            "learning_rate": 0.001,
            "d_block": 64,
            "n_blocks": 3,
            "ffn_d_hidden": 128,
            "attention_n_heads": 4,

            # Keep attention dropout unchanged
            "attention_dropout": 0.1,

            # Variable being tested
            "ffn_dropout": ffn_dropout,

            # Keep residual dropout unchanged
            "residual_dropout": 0.0,

            # Reproducibility
            "seed": 42,

            # Training controls
            "max_epochs": 100,
            "patience": 15,
            "validation_fraction_inner": 0.15,
            "scheduler_factor": 0.5,
            "scheduler_patience": 5,
            "min_delta": 1e-6,
        }
    )


    candidate_adapter = FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=candidate_params,
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


    start_time = time.perf_counter()


    candidate_adapter.fit(
        X=X_outer_train,
        y=y_outer_train,
        epochs=None,
        batch_size=512,
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    metadata = (
        candidate_adapter
        .get_metadata()
    )


    ffn_dropout_results.append(
        {
            "ffn_dropout": ffn_dropout,

            "best_inner_pr_auc": (
                metadata[
                    "best_inner_pr_auc"
                ]
            ),

            "best_epoch": (
                metadata[
                    "best_epoch"
                ]
            ),

            "epochs_trained": (
                metadata[
                    "epochs_trained"
                ]
            ),

            "pos_weight": (
                metadata[
                    "pos_weight"
                ]
            ),

            "training_minutes": (
                elapsed_seconds / 60
            ),
        }
    )


    print(
        "  Best inner PR-AUC:",
        f"{metadata['best_inner_pr_auc'] * 100:.2f}%"
    )

    print(
        "  Best epoch:",
        metadata["best_epoch"]
    )

    print(
        "  Epochs trained:",
        metadata["epochs_trained"]
    )

    print(
        "  Time:",
        f"{elapsed_seconds / 60:.2f} min"
    )


# --------------------------------------------------
# Compare candidates
# --------------------------------------------------

ffn_dropout_screen_df = pd.DataFrame(
    ffn_dropout_results
)

ffn_dropout_screen_df = (
    ffn_dropout_screen_df
    .sort_values(
        "best_inner_pr_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nFFN dropout screening results"
)

display(
    ffn_dropout_screen_df
)


best_ffn_dropout = float(
    ffn_dropout_screen_df.iloc[0][
        "ffn_dropout"
    ]
)


print(
    "\nBest ffn_dropout by inner-validation PR-AUC:",
    best_ffn_dropout
)

print(
    "Best inner-validation PR-AUC:",
    f"{ffn_dropout_screen_df.iloc[0]['best_inner_pr_auc'] * 100:.2f}%"
)


Testing ffn_dropout: 0.0
  Best inner PR-AUC: 10.62%
  Best epoch: 14
  Epochs trained: 29
  Time: 2.00 min

Testing ffn_dropout: 0.1
  Best inner PR-AUC: 10.76%
  Best epoch: 22
  Epochs trained: 37
  Time: 3.57 min

Testing ffn_dropout: 0.2
  Best inner PR-AUC: 10.54%
  Best epoch: 13
  Epochs trained: 28
  Time: 2.73 min

FFN dropout screening results


,ffn_dropout,best_inner_pr_auc,best_epoch,epochs_trained,pos_weight,training_minutes
0,0.1,0.107563,22,37,8.988983,3.571208
1,0.0,0.106223,14,29,8.988983,1.996577
2,0.2,0.105436,13,28,8.988983,2.729680



Best ffn_dropout by inner-validation PR-AUC: 0.1
Best inner-validation PR-AUC: 10.76%


In [22]:
# -----------------------------
# FT-Transformer tuning summary
# -----------------------------
#
# Consolidate the controlled screening results and
# record the final retained configuration.
# -----------------------------

tuning_summary = pd.DataFrame(
    [
        {
            "parameter": "learning_rate",
            "tested": "0.0001, 0.0003, 0.001",
            "retained_value": 0.001,
            "selection_evidence": (
                "Best multi-seed mean inner PR-AUC: 10.66%"
            ),
        },
        {
            "parameter": "d_block",
            "tested": "32, 64, 128",
            "retained_value": 64,
            "selection_evidence": (
                "Highest seed-42 inner PR-AUC: 10.76%"
            ),
        },
        {
            "parameter": "n_blocks",
            "tested": "2, 3, 4",
            "retained_value": 3,
            "selection_evidence": (
                "Best multi-seed mean inner PR-AUC: 10.66%"
            ),
        },
        {
            "parameter": "ffn_d_hidden",
            "tested": "64, 128, 256",
            "retained_value": 128,
            "selection_evidence": (
                "Best multi-seed mean inner PR-AUC: 10.66%"
            ),
        },
        {
            "parameter": "attention_n_heads",
            "tested": "2, 4, 8",
            "retained_value": 4,
            "selection_evidence": (
                "Highest seed-42 inner PR-AUC: 10.76%"
            ),
        },
        {
            "parameter": "ffn_dropout",
            "tested": "0.0, 0.1, 0.2",
            "retained_value": 0.1,
            "selection_evidence": (
                "Highest seed-42 inner PR-AUC: 10.76%"
            ),
        },
    ]
)


display(
    tuning_summary
)


final_ft_params = {
    "learning_rate": 0.001,
    "d_block": 64,
    "n_blocks": 3,
    "ffn_d_hidden": 128,
    "attention_n_heads": 4,
    "ffn_dropout": 0.1,
    "attention_dropout": 0.1,
    "residual_dropout": 0.0,
    "max_epochs": 100,
    "patience": 15,
    "validation_fraction_inner": 0.15,
    "scheduler_factor": 0.5,
    "scheduler_patience": 5,
    "seed": 42,
}


print("\nFinal retained FT-Transformer configuration")
print("-------------------------------------------")

for key, value in final_ft_params.items():
    print(
        f"{key}: {value}"
    )


print("\nPreviously observed untouched Fold 1 result")
print("-------------------------------------------")

print("PR-AUC: 9.66%")
print("ROC-AUC: 48.59%")
print("F1@0.5: 14.96%")
print("Balanced accuracy: 49.14%")
print("Brier score: 0.242199")


print(
    "\nConclusion:"
)

print(
    "Controlled hyperparameter screening did not identify "
    "a configuration that consistently improved on the "
    "original FT-Transformer architecture."
)

,parameter,tested,retained_value,selection_evidence
0,learning_rate,"0.0001, 0.0003, 0.001",0.001,Best multi-seed mean inner PR-AUC: 10.66%
1,d_block,"32, 64, 128",64.000,Highest seed-42 inner PR-AUC: 10.76%
2,n_blocks,"2, 3, 4",3.000,Best multi-seed mean inner PR-AUC: 10.66%
3,ffn_d_hidden,"64, 128, 256",128.000,Best multi-seed mean inner PR-AUC: 10.66%
4,attention_n_heads,"2, 4, 8",4.000,Highest seed-42 inner PR-AUC: 10.76%
5,ffn_dropout,"0.0, 0.1, 0.2",0.100,Highest seed-42 inner PR-AUC: 10.76%



Final retained FT-Transformer configuration
-------------------------------------------
learning_rate: 0.001
d_block: 64
n_blocks: 3
ffn_d_hidden: 128
attention_n_heads: 4
ffn_dropout: 0.1
attention_dropout: 0.1
residual_dropout: 0.0
max_epochs: 100
patience: 15
validation_fraction_inner: 0.15
scheduler_factor: 0.5
scheduler_patience: 5
seed: 42

Previously observed untouched Fold 1 result
-------------------------------------------
PR-AUC: 9.66%
ROC-AUC: 48.59%
F1@0.5: 14.96%
Balanced accuracy: 49.14%
Brier score: 0.242199

Conclusion:
Controlled hyperparameter screening did not identify a configuration that consistently improved on the original FT-Transformer architecture.


In [23]:
# -----------------------------
# Final FT-Transformer 5-fold OOF experiment
# -----------------------------
#
# Final retained architecture:
# - class-weighted BCE loss
# - 15% inner validation
# - PR-AUC early stopping
# - ReduceLROnPlateau
# - best-epoch restoration
#
# The formal 20,000-row holdout remains untouched.
# -----------------------------

import time


# --------------------------------------------------
# Build final parameter set
# --------------------------------------------------
#
# Start from the repository configuration so values
# such as weight_decay are retained, then apply the
# final training controls established above.

final_run_params = fixed_params.copy()

final_run_params.update(
    {
        "learning_rate": 0.001,
        "d_block": 64,
        "n_blocks": 3,
        "ffn_d_hidden": 128,
        "attention_n_heads": 4,

        "ffn_dropout": 0.1,
        "attention_dropout": 0.1,
        "residual_dropout": 0.0,

        "max_epochs": 100,
        "patience": 15,
        "validation_fraction_inner": 0.15,

        "scheduler_factor": 0.5,
        "scheduler_patience": 5,

        "min_delta": 1e-6,

        "seed": 42,
    }
)


# --------------------------------------------------
# Fresh adapter for every outer fold
# --------------------------------------------------

def make_final_ft_transformer():

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=final_run_params.copy(),
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


# --------------------------------------------------
# Pre-run information
# --------------------------------------------------

print(
    "Final weighted + early-stopping FT-Transformer CV"
)

print(
    "-------------------------------------------------"
)

print(
    "Development rows:",
    len(development_df)
)

print(
    "Formal holdout rows used: 0"
)

print(
    "Predictors:",
    len(primary_model_features)
)

print(
    "Outer folds: 5"
)

print(
    "Outer training rows/fold: 64,000"
)

print(
    "Outer validation rows/fold: 16,000"
)

print(
    "Maximum epochs:",
    final_run_params["max_epochs"]
)

print(
    "Early-stopping patience:",
    final_run_params["patience"]
)

print(
    "Inner validation fraction:",
    final_run_params[
        "validation_fraction_inner"
    ]
)

print(
    "Primary early-stopping metric: PR-AUC"
)

print(
    "Class weighting: fold-local positive-class weight"
)

print(
    "Device:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)


# --------------------------------------------------
# Run frozen 5-fold OOF evaluation
# --------------------------------------------------

final_cv_start = time.perf_counter()


final_ft_cv_result = run_5fold_cv(
    adapter_factory=make_final_ft_transformer,

    data=development_df,

    split_registry=split_registry,

    feature_cols=primary_model_features,

    target_col=target_column,

    id_col="Employee_ID",

    fold_col="cv_fold",

    partition_col="partition",

    development_label="development",

    holdout_label="holdout",

    expected_folds=(
        1,
        2,
        3,
        4,
        5,
    ),

    expected_model_name=model_name,
)


final_cv_elapsed = (
    time.perf_counter()
    - final_cv_start
)


# --------------------------------------------------
# Basic integrity checks
# --------------------------------------------------

assert (
    len(
        final_ft_cv_result.predictions
    )
    == 80_000
)

assert (
    final_ft_cv_result
    .predictions["Employee_ID"]
    .is_unique
)

assert set(
    final_ft_cv_result
    .predictions["cv_fold"]
    .unique()
) == {
    1,
    2,
    3,
    4,
    5,
}


# --------------------------------------------------
# Results
# --------------------------------------------------

print(
    "\nFinal 5-fold FT-Transformer run completed."
)

print(
    "Elapsed time:",
    f"{final_cv_elapsed / 60:.2f} minutes"
)

print(
    "OOF prediction rows:",
    len(
        final_ft_cv_result.predictions
    )
)


print(
    "\nPooled OOF metrics"
)

print(
    "------------------"
)

for key, value in (
    final_ft_cv_result
    .summary_metrics
    .items()
):

    print(
        f"{key}: {value}"
    )


print(
    "\nFold-level metrics"
)

display(
    final_ft_cv_result.fold_metrics
)

Final weighted + early-stopping FT-Transformer CV
-------------------------------------------------
Development rows: 80000
Formal holdout rows used: 0
Predictors: 17
Outer folds: 5
Outer training rows/fold: 64,000
Outer validation rows/fold: 16,000
Maximum epochs: 100
Early-stopping patience: 15
Inner validation fraction: 0.15
Primary early-stopping metric: PR-AUC
Class weighting: fold-local positive-class weight
Device: NVIDIA GeForce RTX 4060 Laptop GPU


KeyError: 'cv_fold'

In [24]:
# -----------------------------
# Diagnose final CV result after KeyError
# -----------------------------

print(
    "final_ft_cv_result exists:",
    "final_ft_cv_result" in globals()
)


if "final_ft_cv_result" in globals():

    print(
        "\nPrediction shape:",
        final_ft_cv_result.predictions.shape
    )

    print(
        "\nPrediction columns:"
    )

    print(
        final_ft_cv_result.predictions.columns.tolist()
    )


    print(
        "\nFirst 5 prediction rows:"
    )

    display(
        final_ft_cv_result.predictions.head()
    )


    print(
        "\nFold metric shape:",
        final_ft_cv_result.fold_metrics.shape
    )

    print(
        "\nFold metric columns:"
    )

    print(
        final_ft_cv_result.fold_metrics.columns.tolist()
    )


    print(
        "\nSummary metrics:"
    )

    print(
        final_ft_cv_result.summary_metrics
    )

final_ft_cv_result exists: False


In [25]:
# -----------------------------
# Diagnose CV input columns
# -----------------------------

print("development_df shape:")
print(development_df.shape)

print("\ndevelopment_df columns:")
print(development_df.columns.tolist())


print("\nsplit_registry shape:")
print(split_registry.shape)

print("\nsplit_registry columns:")
print(split_registry.columns.tolist())


print("\nDoes development_df already contain split columns?")

for column in [
    "Employee_ID",
    "cv_fold",
    "partition",
]:
    print(
        f"{column}:",
        column in development_df.columns
    )

development_df shape:
(80000, 22)

development_df columns:
['Employee_ID', 'Department', 'Gender', 'Age', 'Job_Title', 'Hire_Date', 'Years_At_Company', 'Education_Level', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned', 'partition', 'cv_fold']

split_registry shape:
(100000, 3)

split_registry columns:
['Employee_ID', 'partition', 'cv_fold']

Does development_df already contain split columns?
Employee_ID: True
cv_fold: True
partition: True


In [26]:
# -----------------------------
# Final FT-Transformer 5-fold OOF experiment
# CORRECTED: pass development_data, not development_df
# -----------------------------

import time


# --------------------------------------------------
# Final retained parameters
# --------------------------------------------------

final_run_params = fixed_params.copy()

final_run_params.update(
    {
        "learning_rate": 0.001,
        "d_block": 64,
        "n_blocks": 3,
        "ffn_d_hidden": 128,
        "attention_n_heads": 4,

        "ffn_dropout": 0.1,
        "attention_dropout": 0.1,
        "residual_dropout": 0.0,

        "max_epochs": 100,
        "patience": 15,
        "validation_fraction_inner": 0.15,

        "scheduler_factor": 0.5,
        "scheduler_patience": 5,

        "min_delta": 1e-6,

        "seed": 42,
    }
)


# --------------------------------------------------
# Fresh adapter for every outer CV fold
# --------------------------------------------------

def make_final_ft_transformer():

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=final_run_params.copy(),
        categorical_features=(
            categorical_feature_names.copy()
        ),
    )


# --------------------------------------------------
# Verify correct runner input
# --------------------------------------------------

assert len(development_data) == 80_000

assert "cv_fold" not in development_data.columns
assert "partition" not in development_data.columns

assert set(
    development_data["Employee_ID"]
).isdisjoint(
    holdout_ids
)


print(
    "Final weighted + early-stopping FT-Transformer CV"
)

print(
    "-------------------------------------------------"
)

print(
    "Development rows:",
    len(development_data)
)

print(
    "Formal holdout rows used: 0"
)

print(
    "Predictors:",
    len(primary_model_features)
)

print(
    "Outer folds: 5"
)

print(
    "Maximum epochs:",
    final_run_params["max_epochs"]
)

print(
    "Early-stopping patience:",
    final_run_params["patience"]
)

print(
    "Inner validation fraction:",
    final_run_params[
        "validation_fraction_inner"
    ]
)

print(
    "Primary early-stopping metric: PR-AUC"
)

print(
    "Class weighting: inner-training positive-class weight"
)

print(
    "Device:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)


# --------------------------------------------------
# Run frozen 5-fold OOF evaluation
# --------------------------------------------------

final_cv_start = time.perf_counter()


final_ft_cv_result = run_5fold_cv(
    adapter_factory=make_final_ft_transformer,

    # IMPORTANT:
    # Raw development rows only.
    # The runner attaches cv_fold from split_registry.
    data=development_data,

    split_registry=split_registry,

    feature_cols=primary_model_features,

    target_col=target_column,

    id_col="Employee_ID",

    fold_col="cv_fold",

    partition_col="partition",

    development_label="development",

    holdout_label="holdout",

    expected_folds=(
        1,
        2,
        3,
        4,
        5,
    ),

    expected_model_name=model_name,
)


final_cv_elapsed = (
    time.perf_counter()
    - final_cv_start
)


# --------------------------------------------------
# Integrity checks
# --------------------------------------------------

assert (
    len(
        final_ft_cv_result.predictions
    )
    == 80_000
)

assert (
    final_ft_cv_result
    .predictions["Employee_ID"]
    .is_unique
)

assert set(
    final_ft_cv_result
    .predictions["cv_fold"]
    .unique()
) == {
    1,
    2,
    3,
    4,
    5,
}

assert set(
    final_ft_cv_result
    .predictions["Employee_ID"]
).isdisjoint(
    holdout_ids
)


# --------------------------------------------------
# Results
# --------------------------------------------------

print(
    "\nFinal 5-fold FT-Transformer run completed."
)

print(
    "Elapsed time:",
    f"{final_cv_elapsed / 60:.2f} minutes"
)

print(
    "OOF prediction rows:",
    len(
        final_ft_cv_result.predictions
    )
)


print(
    "\nPooled OOF metrics"
)

print(
    "------------------"
)

for key, value in (
    final_ft_cv_result
    .summary_metrics
    .items()
):

    print(
        f"{key}: {value}"
    )


print(
    "\nFold-level metrics"
)

display(
    final_ft_cv_result.fold_metrics
)

Final weighted + early-stopping FT-Transformer CV
-------------------------------------------------
Development rows: 80000
Formal holdout rows used: 0
Predictors: 17
Outer folds: 5
Maximum epochs: 100
Early-stopping patience: 15
Inner validation fraction: 0.15
Primary early-stopping metric: PR-AUC
Class weighting: inner-training positive-class weight
Device: NVIDIA GeForce RTX 4060 Laptop GPU

Final 5-fold FT-Transformer run completed.
Elapsed time: 3.97 minutes
OOF prediction rows: 80000

Pooled OOF metrics
------------------
model_name: FT-Transformer
model_id: ft_transformer
total_development_records: 80000
total_folds: 5
pooled_average_precision: 0.09997911344857369
pooled_roc_auc: 0.49725447661850314
pooled_f1: 0.16725777849427698
pooled_brier_score: 0.2474131275737539
cv_mean_average_precision: 0.09969554215923253
cv_std_average_precision: 0.0022368687297277495
cv_mean_roc_auc: 0.49757034014440843
cv_std_roc_auc: 0.0065971561882389885
cv_mean_f1: 0.13551815708749912
cv_std_f1: 0

,average_precision,roc_auc,precision,recall,f1,balanced_accuracy,accuracy,brier_score,tn,fp,fn,tp,predicted_positive_count,predicted_positive_rate,fold,fit_time_seconds,predict_time_seconds
0,0.096579,0.485936,0.095721,0.342286,0.149604,0.491374,0.610625,0.242199,9222,5177,1053,548,5725,0.357812,1,58.556671,0.201329
1,0.102827,0.501306,0.100974,0.544035,0.170333,0.502728,0.469687,0.255291,6644,7755,730,871,8626,0.539125,2,83.212348,0.186282
2,0.100136,0.500504,0.000000,0.000000,0.000000,0.500000,0.899875,0.217159,14398,0,1602,0,0,0.000000,3,36.644393,0.192728
3,0.099155,0.498664,0.099760,0.726592,0.175433,0.498523,0.316125,0.261640,3894,10504,438,1164,11668,0.729250,4,35.025116,0.183175
4,0.099781,0.501442,0.100468,0.978152,0.182220,0.501856,0.120938,0.260776,368,14030,35,1567,15597,0.974812,5,23.281531,0.210745


In [27]:
# -----------------------------
# Save final weighted + early-stopping FT-Transformer experiment
# -----------------------------

import json
from pathlib import Path


FINAL_ES_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
    / "weighted_early_stopping_final"
)

FINAL_ES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# 1. OOF predictions
# --------------------------------------------------

oof_path = (
    FINAL_ES_DIR
    / "ft_transformer_final_oof_predictions.csv"
)

final_ft_cv_result.predictions.to_csv(
    oof_path,
    index=False,
)


# --------------------------------------------------
# 2. Fold-level metrics
# --------------------------------------------------

fold_metrics_path = (
    FINAL_ES_DIR
    / "ft_transformer_final_fold_metrics.csv"
)

final_ft_cv_result.fold_metrics.to_csv(
    fold_metrics_path,
    index=False,
)


# --------------------------------------------------
# 3. Summary metrics
# --------------------------------------------------

summary_path = (
    FINAL_ES_DIR
    / "ft_transformer_final_summary_metrics.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_ft_cv_result.summary_metrics,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# 4. Final configuration
# --------------------------------------------------

config_path = (
    FINAL_ES_DIR
    / "ft_transformer_final_configuration.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_run_params,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# 5. Experiment metadata
# --------------------------------------------------

metadata = {
    "model": "FT-Transformer",
    "experiment": "weighted_early_stopping_final",

    "development_rows": 80000,
    "formal_holdout_rows_used": 0,
    "outer_cv_folds": 5,

    "class_weighting": (
        "inner-training positive-class weight"
    ),

    "pos_weight_formula": (
        "negative_count / positive_count"
    ),

    "inner_validation_fraction": 0.15,
    "early_stopping_metric": "PR-AUC",
    "maximum_epochs": 100,
    "patience": 15,

    "learning_rate_scheduler": (
        "ReduceLROnPlateau"
    ),

    "best_weights_restored": True,

    "primary_metric": "PR-AUC",

    "pooled_pr_auc": (
        final_ft_cv_result
        .summary_metrics[
            "pooled_average_precision"
        ]
    ),

    "pooled_roc_auc": (
        final_ft_cv_result
        .summary_metrics[
            "pooled_roc_auc"
        ]
    ),

    "elapsed_minutes": (
        final_cv_elapsed / 60
    ),

    "conclusion": (
        "Weighted FT-Transformer with inner-validation "
        "early stopping remained approximately at no-skill "
        "performance. Further architecture tuning was stopped "
        "because controlled screening did not produce a "
        "consistent PR-AUC improvement."
    ),
}


metadata_path = (
    FINAL_ES_DIR
    / "ft_transformer_final_run_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# Verification
# --------------------------------------------------

saved_files = [
    oof_path,
    fold_metrics_path,
    summary_path,
    config_path,
    metadata_path,
]


assert len(
    final_ft_cv_result.predictions
) == 80_000

assert (
    final_ft_cv_result
    .predictions["Employee_ID"]
    .is_unique
)

assert set(
    final_ft_cv_result
    .predictions["Employee_ID"]
).isdisjoint(
    holdout_ids
)

assert all(
    path.exists()
    for path in saved_files
)


print(
    "Final FT-Transformer experiment saved."
)

print(
    "\nDirectory:"
)

print(
    FINAL_ES_DIR
)

print(
    "\nSaved files:"
)

for path in saved_files:

    print(
        " ✓",
        path.name,
        f"({path.stat().st_size:,} bytes)"
    )


print(
    "\nFinal artifact integrity checks PASSED."
)

Final FT-Transformer experiment saved.

Directory:
D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\weighted_early_stopping_final

Saved files:
 ✓ ft_transformer_final_oof_predictions.csv (3,760,505 bytes)
 ✓ ft_transformer_final_fold_metrics.csv (1,210 bytes)
 ✓ ft_transformer_final_summary_metrics.json (710 bytes)
 ✓ ft_transformer_final_configuration.json (400 bytes)
 ✓ ft_transformer_final_run_metadata.json (938 bytes)

Final artifact integrity checks PASSED.


## Final FT-Transformer Results and Conclusion

The FT-Transformer investigation was completed using the frozen 80,000-row development partition and 5-fold cross-validation. The 20,000-row formal holdout set remained completely untouched during model development, weighting, early stopping, threshold analysis, and hyperparameter screening.

### Experimental Progression

Three main FT-Transformer experiments were evaluated.

| Experiment | PR-AUC | ROC-AUC | Main Observation |
|---|---:|---:|---|
| Unweighted fixed training | 10.11% | 49.88% | Model produced almost no positive predictions at the default 0.5 threshold |
| Weighted fixed training | 9.88% | 49.57% | Recall and F1 increased, but ranking performance did not improve |
| Weighted + early stopping | **10.00%** | **49.73%** | More controlled training, but discrimination remained approximately no-skill |

The resignation prevalence in the development data is approximately **10.01%**. Therefore, a PR-AUC close to 10% represents approximately no-skill ranking performance.

---

### Class-Imbalance Handling

Because resignation is the minority class, positive-class weighting was introduced using:

`pos_weight = number of negative training samples / number of positive training samples`

The weight was calculated independently from the training data available inside each fold.

The weighted loss used:

`BCEWithLogitsLoss(pos_weight=pos_weight_tensor)`

This increased the model's willingness to predict the resignation class and improved recall and F1 at some thresholds.

However, class weighting did not improve PR-AUC or ROC-AUC.

---

### Threshold Analysis

Threshold analysis was performed using development out-of-fold predictions only.

The mathematically highest F1-score occurred at a threshold of approximately `0.05`.

At this threshold:

- Precision = approximately 10.01%
- Recall = 100%
- F1 = approximately 18.20%
- Balanced accuracy = 50%
- Predicted positive rate = 100%

This threshold classified every employee as a potential resigner.

Therefore, the higher F1-score represented a trivial all-positive classifier rather than useful predictive discrimination.

Threshold adjustment was consequently not used as a solution to the model's weak performance.

---

### Early Stopping and Training Improvements

The final training procedure introduced:

- 15% stratified inner validation inside each outer training fold
- Maximum of 100 epochs
- PR-AUC-based model monitoring
- Early stopping with patience of 15 epochs
- `ReduceLROnPlateau`
- Restoration of the model weights from the best inner-validation epoch
- Fold-local preprocessing
- Inner-training positive-class weighting

For example, during the initial full Fold 1 test:

- Best epoch = 22
- Training stopped after 37 epochs
- Best inner-validation PR-AUC = 10.76%
- Untouched outer Fold 1 PR-AUC = 9.66%

This demonstrated that early stopping operated correctly, but the stronger inner-validation result did not translate into improved outer-fold performance.

---

### Hyperparameter Screening

A controlled hyperparameter investigation was performed using inner-validation PR-AUC.

The following parameters were examined:

- Learning rate
- Transformer block dimension
- Number of Transformer blocks
- Feed-forward hidden dimension
- Number of attention heads
- Feed-forward dropout

Where single-seed differences appeared potentially meaningful, the strongest candidates were re-evaluated across multiple random seeds.

The final retained configuration was:

| Parameter | Final Value |
|---|---:|
| Learning rate | 0.001 |
| Transformer blocks | 3 |
| Block dimension | 64 |
| Attention heads | 4 |
| FFN hidden dimension | 128 |
| FFN dropout | 0.1 |
| Attention dropout | 0.1 |
| Residual dropout | 0.0 |
| Maximum epochs | 100 |
| Early-stopping patience | 15 |
| Inner validation fraction | 0.15 |

Controlled screening did not identify an alternative configuration that produced a consistent improvement over this architecture.

---

### Final 5-Fold OOF Result

The final weighted + early-stopping FT-Transformer generated predictions for all **80,000 development employees**.

The pooled out-of-fold results were:

- **PR-AUC: 9.998%**
- **ROC-AUC: 49.73%**

Both metrics remain approximately at no-skill levels.

Classification behaviour at the default 0.5 threshold also varied substantially between folds. For example:

- Fold 3 predicted no employees as positive.
- Fold 5 predicted approximately 97.48% of employees as positive.

This instability further indicates that the weighted raw probabilities should not be interpreted directly as reliable resignation-risk probabilities.

---

### Final Interpretation

The experiments show that class imbalance was responsible for part of the initial threshold behaviour, particularly the zero F1-score produced by the unweighted model.

Positive-class weighting successfully increased sensitivity to the minority resignation class.

However, weighting, threshold adjustment, early stopping, learning-rate scheduling, and controlled architecture tuning did not produce meaningful improvements in the primary ranking metric.

The final PR-AUC remained approximately equal to the resignation prevalence, while ROC-AUC remained approximately 0.50.

Therefore, the main limitation of the FT-Transformer is unlikely to be class imbalance or the default decision threshold alone.

The available 17 predictors appear to provide very limited discriminatory signal for predicting resignation with this model under the current project design.

Further FT-Transformer hyperparameter searching was therefore stopped to avoid excessive tuning to the development data.

The final weighted + early-stopping experiment is retained as the principal FT-Transformer result, while the unweighted and weighted fixed experiments are preserved as comparison baselines.

The formal holdout remains untouched for later locked final-model evaluation.

In [28]:
# -----------------------------
# Save FT-Transformer tuning evidence
# -----------------------------
#
# Preserve all controlled hyperparameter screening
# and robustness-check tables used to justify the
# final retained configuration.
# -----------------------------

from pathlib import Path


TUNING_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
    / "weighted_early_stopping_final"
    / "tuning"
)

TUNING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Tuning tables expected from this notebook session
# --------------------------------------------------

tuning_artifacts = {
    "learning_rate_screen.csv":
        lr_screen_df,

    "learning_rate_robustness_detailed.csv":
        lr_robustness_df,

    "learning_rate_robustness_summary.csv":
        lr_robustness_summary,

    "d_block_screen.csv":
        d_block_screen_df,

    "n_blocks_screen.csv":
        n_block_screen_df,

    "n_blocks_robustness_detailed.csv":
        n_block_robustness_df,

    "n_blocks_robustness_summary.csv":
        n_block_robustness_summary,

    "ffn_hidden_screen.csv":
        ffn_hidden_screen_df,

    "ffn_hidden_robustness_detailed.csv":
        ffn_robustness_df,

    "ffn_hidden_robustness_summary.csv":
        ffn_robustness_summary,

    "attention_heads_screen.csv":
        attention_head_screen_df,

    "ffn_dropout_screen.csv":
        ffn_dropout_screen_df,

    "final_tuning_summary.csv":
        tuning_summary,
}


# --------------------------------------------------
# Save tables
# --------------------------------------------------

saved_tuning_files = []


for filename, dataframe in tuning_artifacts.items():

    output_path = (
        TUNING_DIR
        / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    saved_tuning_files.append(
        output_path
    )


# --------------------------------------------------
# Verification
# --------------------------------------------------

assert len(saved_tuning_files) == len(
    tuning_artifacts
)

assert all(
    path.exists()
    for path in saved_tuning_files
)


print(
    "FT-Transformer tuning evidence saved."
)

print(
    "\nDirectory:"
)

print(
    TUNING_DIR
)

print(
    "\nSaved files:"
)

for path in saved_tuning_files:

    print(
        " ✓",
        path.name,
        f"({path.stat().st_size:,} bytes)"
    )


print(
    "\nTuning artifact checks PASSED."
)

FT-Transformer tuning evidence saved.

Directory:
D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\weighted_early_stopping_final\tuning

Saved files:
 ✓ learning_rate_screen.csv (298 bytes)
 ✓ learning_rate_robustness_detailed.csv (416 bytes)
 ✓ learning_rate_robustness_summary.csv (316 bytes)
 ✓ d_block_screen.csv (280 bytes)
 ✓ n_blocks_screen.csv (278 bytes)
 ✓ n_blocks_robustness_detailed.csv (380 bytes)
 ✓ n_blocks_robustness_summary.csv (286 bytes)
 ✓ ffn_hidden_screen.csv (289 bytes)
 ✓ ffn_hidden_robustness_detailed.csv (398 bytes)
 ✓ ffn_hidden_robustness_summary.csv (296 bytes)
 ✓ attention_heads_screen.csv (285 bytes)
 ✓ ffn_dropout_screen.csv (287 bytes)
 ✓ final_tuning_summary.csv (487 bytes)

Tuning artifact checks PASSED.


In [29]:
# -----------------------------
# FT-Transformer consistency check
# -----------------------------
#
# READ-ONLY:
# This cell does not modify repository files.
# -----------------------------

import py_compile
from pathlib import Path


# --------------------------------------------------
# Important paths
# --------------------------------------------------

ft_source_path = (
    PROJECT_ROOT
    / "src"
    / "models"
    / "ft_transformer.py"
)

canonical_notebook_path = (
    PROJECT_ROOT
    / "notebooks"
    / "05_member_1"
    / "05_m1_ft_transformer.ipynb"
)

ft_results_root = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

unweighted_dir = (
    ft_results_root
    / "unweighted_baseline"
)

weighted_fixed_dir = (
    ft_results_root
    / "weighted_fixed_baseline"
)

final_dir = (
    ft_results_root
    / "weighted_early_stopping_final"
)

tuning_dir = (
    final_dir
    / "tuning"
)


# --------------------------------------------------
# 1. Required repository files
# --------------------------------------------------

assert ft_source_path.exists()
assert canonical_notebook_path.exists()

print("Repository files")
print("----------------")
print("✓ FT-Transformer source exists")
print("✓ Canonical notebook exists")


# --------------------------------------------------
# 2. Python syntax check
# --------------------------------------------------

py_compile.compile(
    str(ft_source_path),
    doraise=True,
)

print("✓ ft_transformer.py syntax valid")


# --------------------------------------------------
# 3. Experiment directories
# --------------------------------------------------

assert unweighted_dir.exists()
assert weighted_fixed_dir.exists()
assert final_dir.exists()
assert tuning_dir.exists()

print("\nExperiment directories")
print("----------------------")
print("✓ unweighted_baseline")
print("✓ weighted_fixed_baseline")
print("✓ weighted_early_stopping_final")
print("✓ tuning evidence directory")


# --------------------------------------------------
# 4. Final experiment files
# --------------------------------------------------

required_final_files = [
    "ft_transformer_final_oof_predictions.csv",
    "ft_transformer_final_fold_metrics.csv",
    "ft_transformer_final_summary_metrics.json",
    "ft_transformer_final_configuration.json",
    "ft_transformer_final_run_metadata.json",
]

for filename in required_final_files:

    path = final_dir / filename

    assert path.exists(), (
        f"Missing final artifact: {filename}"
    )

print(
    f"\n✓ All {len(required_final_files)} "
    "final experiment files exist"
)


# --------------------------------------------------
# 5. Tuning evidence
# --------------------------------------------------

expected_tuning_files = [
    "learning_rate_screen.csv",
    "learning_rate_robustness_detailed.csv",
    "learning_rate_robustness_summary.csv",
    "d_block_screen.csv",
    "n_blocks_screen.csv",
    "n_blocks_robustness_detailed.csv",
    "n_blocks_robustness_summary.csv",
    "ffn_hidden_screen.csv",
    "ffn_hidden_robustness_detailed.csv",
    "ffn_hidden_robustness_summary.csv",
    "attention_heads_screen.csv",
    "ffn_dropout_screen.csv",
    "final_tuning_summary.csv",
]

for filename in expected_tuning_files:

    assert (
        tuning_dir
        / filename
    ).exists(), (
        f"Missing tuning artifact: {filename}"
    )

print(
    f"✓ All {len(expected_tuning_files)} "
    "tuning evidence files exist"
)


# --------------------------------------------------
# 6. Final OOF result integrity
# --------------------------------------------------

final_oof = (
    final_ft_cv_result
    .predictions
)

assert len(final_oof) == 80_000

assert (
    final_oof["Employee_ID"]
    .is_unique
)

assert set(
    final_oof["cv_fold"].unique()
) == {
    1,
    2,
    3,
    4,
    5,
}

assert set(
    final_oof["Employee_ID"]
).isdisjoint(
    holdout_ids
)

print("\nFinal OOF integrity")
print("-------------------")
print("✓ 80,000 OOF predictions")
print("✓ Employee_ID unique")
print("✓ folds 1–5 represented")
print("✓ zero formal-holdout IDs")


# --------------------------------------------------
# 7. Final parameter consistency
# --------------------------------------------------

expected_final_params = {
    "learning_rate": 0.001,
    "d_block": 64,
    "n_blocks": 3,
    "ffn_d_hidden": 128,
    "attention_n_heads": 4,
    "ffn_dropout": 0.1,
    "attention_dropout": 0.1,
    "residual_dropout": 0.0,
    "max_epochs": 100,
    "patience": 15,
    "validation_fraction_inner": 0.15,
    "seed": 42,
}


for key, expected_value in (
    expected_final_params.items()
):

    actual_value = final_run_params[key]

    assert actual_value == expected_value, (
        f"{key}: expected {expected_value}, "
        f"found {actual_value}"
    )


print("\nFinal configuration")
print("-------------------")

for key, value in expected_final_params.items():
    print(
        f"✓ {key} = {value}"
    )


print(
    "\nFT-Transformer consistency checks PASSED."
)

Repository files
----------------
✓ FT-Transformer source exists
✓ Canonical notebook exists
✓ ft_transformer.py syntax valid

Experiment directories
----------------------
✓ unweighted_baseline
✓ weighted_fixed_baseline
✓ weighted_early_stopping_final
✓ tuning evidence directory

✓ All 5 final experiment files exist
✓ All 13 tuning evidence files exist

Final OOF integrity
-------------------
✓ 80,000 OOF predictions
✓ Employee_ID unique
✓ folds 1–5 represented
✓ zero formal-holdout IDs

Final configuration
-------------------
✓ learning_rate = 0.001
✓ d_block = 64
✓ n_blocks = 3
✓ ffn_d_hidden = 128
✓ attention_n_heads = 4
✓ ffn_dropout = 0.1
✓ attention_dropout = 0.1
✓ residual_dropout = 0.0
✓ max_epochs = 100
✓ patience = 15
✓ validation_fraction_inner = 0.15
✓ seed = 42

FT-Transformer consistency checks PASSED.


In [30]:
# -----------------------------
# Canonical FT-Transformer notebook content audit
# -----------------------------
#
# READ-ONLY.
#
# Purpose:
# Check the saved notebook for stale statements,
# duplicated experiment calls, debugging remnants,
# and references that may conflict with the final
# weighted + early-stopping methodology.
# -----------------------------

import json
from pathlib import Path


notebook_path = (
    PROJECT_ROOT
    / "notebooks"
    / "05_member_1"
    / "05_m1_ft_transformer.ipynb"
)


# --------------------------------------------------
# Load saved notebook
# --------------------------------------------------

with open(
    notebook_path,
    "r",
    encoding="utf-8",
) as file:

    notebook_json = json.load(file)


cells = notebook_json["cells"]

code_cells = [
    cell
    for cell in cells
    if cell["cell_type"] == "code"
]

markdown_cells = [
    cell
    for cell in cells
    if cell["cell_type"] == "markdown"
]


def cell_text(cell):
    return "".join(
        cell.get("source", [])
    )


all_code = "\n".join(
    cell_text(cell)
    for cell in code_cells
)

all_markdown = "\n".join(
    cell_text(cell)
    for cell in markdown_cells
)

all_content = (
    all_code
    + "\n"
    + all_markdown
)


# --------------------------------------------------
# Basic structure
# --------------------------------------------------

print("Canonical notebook audit")
print("------------------------")

print(
    "Notebook:",
    notebook_path
)

print(
    "Total cells:",
    len(cells)
)

print(
    "Code cells:",
    len(code_cells)
)

print(
    "Markdown cells:",
    len(markdown_cells)
)


# --------------------------------------------------
# Important final-methodology evidence
# --------------------------------------------------

required_terms = {
    "class weighting":
        "pos_weight",

    "inner validation":
        "validation_fraction_inner",

    "early stopping":
        "early stopping",

    "PR-AUC":
        "PR-AUC",

    "final weighted experiment":
        "weighted_early_stopping_final",

    "formal holdout":
        "holdout",
}


print(
    "\nFinal methodology evidence"
)

print(
    "--------------------------"
)


for label, term in required_terms.items():

    present = (
        term.lower()
        in all_content.lower()
    )

    print(
        f"{'✓' if present else '✗'} "
        f"{label}: {present}"
    )


# --------------------------------------------------
# Potential stale / cleanup indicators
# --------------------------------------------------

stale_terms = [
    "epochs=50",
    "epochs = 50",
    "50 training epochs",
    "no early stopping",
    "without early stopping",
    "unweighted model is the final",
    "weighted fixed model is the final",
    "inspect.getsource",
    "KeyboardInterrupt",
]


print(
    "\nPotential stale/debug references"
)

print(
    "--------------------------------"
)


for term in stale_terms:

    count = (
        all_content
        .lower()
        .count(
            term.lower()
        )
    )

    print(
        f"{term!r}: {count}"
    )


# --------------------------------------------------
# Locate run_5fold_cv calls
# --------------------------------------------------

run_call_cells = []


for index, cell in enumerate(cells):

    if (
        cell["cell_type"] == "code"
        and "run_5fold_cv(" in cell_text(cell)
    ):

        run_call_cells.append(
            index
        )


print(
    "\nrun_5fold_cv calls"
)

print(
    "------------------"
)

print(
    "Count:",
    len(run_call_cells)
)

print(
    "Cell indexes:",
    run_call_cells
)


# --------------------------------------------------
# Locate experiment result terminology
# --------------------------------------------------

experiment_terms = [
    "unweighted_baseline",
    "weighted_fixed_baseline",
    "weighted_early_stopping_final",
    "final_ft_cv_result",
    "weighted_ft_cv_result",
]


print(
    "\nExperiment references"
)

print(
    "---------------------"
)


for term in experiment_terms:

    count = (
        all_content.count(
            term
        )
    )

    print(
        f"{term}: {count}"
    )


# --------------------------------------------------
# Check formal holdout safety language
# --------------------------------------------------

holdout_safety_phrases = [
    "formal holdout remains untouched",
    "formal holdout set remained",
    "formal holdout rows used: 0",
]


holdout_safety_found = any(
    phrase.lower()
    in all_content.lower()

    for phrase
    in holdout_safety_phrases
)


print(
    "\nHoldout documentation"
)

print(
    "---------------------"
)

print(
    "✓ Holdout safety statement found"
    if holdout_safety_found
    else "✗ Holdout safety statement not found"
)


print(
    "\nNotebook content audit completed."
)

print(
    "No files were modified."
)

Canonical notebook audit
------------------------
Notebook: D:\Desktop\Employee_Resignation_Prediction\notebooks\05_member_1\05_m1_ft_transformer.ipynb
Total cells: 34
Code cells: 27
Markdown cells: 7

Final methodology evidence
--------------------------
✓ class weighting: True
✗ inner validation: False
✓ early stopping: True
✓ PR-AUC: True
✗ final weighted experiment: False
✓ formal holdout: True

Potential stale/debug references
--------------------------------
'epochs=50': 0
'epochs = 50': 0
'50 training epochs': 1
'no early stopping': 0
'without early stopping': 0
'unweighted model is the final': 0
'weighted fixed model is the final': 0
'inspect.getsource': 0
'KeyboardInterrupt': 0

run_5fold_cv calls
------------------
Count: 2
Cell indexes: [12, 27]

Experiment references
---------------------
unweighted_baseline: 1
weighted_fixed_baseline: 2
weighted_early_stopping_final: 0
final_ft_cv_result: 0
weighted_ft_cv_result: 17

Holdout documentation
---------------------
✓ Holdout sa

In [31]:
# -----------------------------
# Locate notebook cells needing cleanup
# -----------------------------
#
# READ-ONLY.
# Shows the exact cells containing old terminology
# or relevant FT-Transformer methodology sections.
# -----------------------------

search_terms = [
    "50 training epochs",
    "weighted_fixed",
    "unweighted",
    "early stopping",
    "inner validation",
    "validation_fraction_inner",
    "weighted_early_stopping_final",
    "final_ft_cv_result",
    "run_5fold_cv(",
]


print("Notebook cleanup locator")
print("------------------------")


for term in search_terms:

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"SEARCH: {term!r}"
    )

    print(
        f"{'=' * 70}"
    )

    matches = []


    for index, cell in enumerate(cells):

        text = cell_text(cell)

        if term.lower() in text.lower():

            matches.append(
                (
                    index,
                    cell["cell_type"],
                    text,
                )
            )


    if not matches:

        print("No matching cells.")

        continue


    for (
        index,
        cell_type,
        text,
    ) in matches:

        # Keep output readable.
        preview = text.strip()

        if len(preview) > 1200:
            preview = (
                preview[:1200]
                + "\n...[truncated]..."
            )

        print(
            f"\nCELL {index}"
        )

        print(
            f"Type: {cell_type}"
        )

        print(
            "-" * 50
        )

        print(
            preview
        )


print(
    "\n\nLocator completed."
)

print(
    "No notebook cells were modified."
)

Notebook cleanup locator
------------------------

SEARCH: '50 training epochs'

CELL 29
Type: markdown
--------------------------------------------------
## FT-Transformer Experiment Progress and Class-Imbalance Handling

The FT-Transformer experiment was developed and evaluated using the project's frozen 5-fold cross-validation setup on the 80,000-row development partition. The 20,000-row formal holdout set remained excluded throughout model development.

### 1. Initial FT-Transformer Setup

The model used all 17 approved predictors:

- 13 numerical / ordinal features
- 4 categorical features

Numerical features were standardized using `StandardScaler`, while categorical features were encoded independently inside each training fold.

A fresh FT-Transformer model and fresh preprocessing objects were created for every cross-validation fold to prevent information leakage.

The model used the following main architecture:

- 3 Transformer blocks
- Block dimension: 64
- 4 attention heads
-

In [32]:
# -----------------------------
# Compact notebook cleanup map
# -----------------------------
#
# READ-ONLY.
# Shows only cell indexes and first lines.
# -----------------------------

search_terms = [
    "50 training epochs",
    "weighted_fixed_baseline",
    "unweighted_baseline",
    "early stopping",
    "validation_fraction_inner",
    "weighted_early_stopping_final",
    "final_ft_cv_result",
    "run_5fold_cv(",
]


print("Compact cleanup map")
print("-------------------")


for term in search_terms:

    print(f"\n{term!r}")

    found = False

    for index, cell in enumerate(cells):

        text = cell_text(cell)

        if term.lower() in text.lower():

            found = True

            first_line = (
                text.strip()
                .splitlines()[0]
                if text.strip()
                else "<empty>"
            )

            print(
                f"  Cell {index:>2} | "
                f"{cell['cell_type']:<8} | "
                f"{first_line[:100]}"
            )

    if not found:
        print("  No matches")


print("\nNo files were modified.")

Compact cleanup map
-------------------

'50 training epochs'
  Cell 29 | markdown | ## FT-Transformer Experiment Progress and Class-Imbalance Handling

'weighted_fixed_baseline'
  Cell 32 | code     | # -----------------------------

'unweighted_baseline'
  Cell 26 | code     | # -----------------------------

'early stopping'
  Cell 10 | markdown | ## Verified FT-Transformer Execution Protocol

'validation_fraction_inner'
  No matches

'weighted_early_stopping_final'
  No matches

'final_ft_cv_result'
  No matches

'run_5fold_cv('
  Cell 12 | code     | # -----------------------------
  Cell 27 | code     | # -----------------------------

No files were modified.
